# HARP — Hybrid Agent Ranking via Personalized PageRank

**A complete, end-to-end notebook implementation of the HARP algorithm for ranking LLM agents against incoming tasks.**

---

## What this notebook is

This notebook is a self-contained, runnable, and fully-explained implementation of **HARP** (Hybrid Agent Ranking via Personalized PageRank), a multi-relational personalized PageRank algorithm for the *agent-routing* problem: given a free-text task and a pool of heterogeneous AI agents (different scaffolds, different model families, different tool kits), which agent should you hand the task to?

HARP fuses **three orthogonal signals** into a single principled score per agent:

1. **Performance history** — Bayesian-smoothed empirical success rates on past tasks.
2. **Endorsement / collaboration graph** — EigenTrust-style co-success signal among agents.
3. **Skill–task semantic similarity** — sentence-embedding cosine similarity between agent self-descriptions and the current task.

These three signals are encoded as three column-stochastic transition matrices over a common state space $V = \mathcal{A} \cup \mathcal{S} \cup \{\tau\}$ (agents, skills, current task). The matrices are combined convexly into a single Markov operator $\mathbf{M}_\tau$, and a **task-conditional softmax teleport vector** $\mathbf{p}_\tau$ is constructed from the task embedding. The algorithm then runs the personalized-PageRank power iteration

$$\mathbf{r}_{t+1} = \alpha \, \mathbf{M}_\tau \, \mathbf{r}_t + (1 - \alpha) \, \mathbf{p}_\tau,$$

which (by the Banach fixed-point theorem) is a contraction in $\ell_1$ with rate $\alpha$. Convergence to a unique stationary distribution is therefore guaranteed.

---

## Why this is interesting

A naive router uses only one signal — "pick the agent with the highest semantic similarity," or "pick the one with the best historical success rate." Each of these breaks badly on its own:

- **Pure semantics** is fooled by agents that *describe themselves* well but actually perform poorly.
- **Pure performance** has no way to handle a task no agent has tried before (cold start).
- **Pure endorsement** rewards popular generalists at the expense of underused specialists.

A *weighted-average* baseline that linearly combines all three signals is a strong competitor — but it discards the *graph structure* connecting agents through shared skills and tasks. HARP fixes that: it does the fusion inside a Markov chain, so the random walker can hop **agent → skill → agent** (transferring credit through shared competence) or **task → skill → agent** (routing through the closest matching skill rather than a blunt direct match).

---

## What you will build

By the end of this notebook you will have:

- A **registry of 12 agents** modeled on real scaffolds from the HAL leaderboard for GAIA.
- A **vocabulary of 8 skills** covering web, PDF, code, vision, math, audio, and spreadsheets.
- A pipeline that **generates synthetic GAIA-style tasks** with the official L1/L2/L3 difficulty mix.
- A **simulated outcome history** that gives HARP something to learn from.
- The three transition matrices $\mathbf{M}^P$, $\mathbf{M}^C$, $\mathbf{M}^\phi$ and the softmax teleport $\mathbf{p}_\tau$.
- The **HARP power iteration** with provable convergence.
- **Five baselines** (random, perf-only, sim-only, vanilla PageRank, weighted-average) for comparison.
- An IR-style **benchmark** with NDCG@5, MRR, P@3, R@5, and Regret metrics.
- An **ablation study** that quantifies how much each signal contributes.
- **Diagnostic plots**: a convergence curve and a per-task agent-score heatmap.
- An **interactive ranking endpoint** you can call on any free-text task.

The whole pipeline runs end-to-end in well under a minute on Colab's free tier.

---

## How to use this notebook

Run the cells **top to bottom**. Each cell builds on the previous ones — there is no skipping ahead. Markdown cells precede each code block and explain both the *why* and the *math* before the *how*.

If you are running this in **Google Colab**, the very first code cell installs all dependencies. If you are running locally, you can use the same line (drop the leading `!`) in a terminal beforehand.

## 1 — The algorithm at a glance

Before the code, here is the entire algorithm on a single page. Everything that follows is just careful unpacking of these few lines.

### 1.1 — State space

Let $\mathcal{A} = \{a_1, \dots, a_n\}$ be the set of agents, $\mathcal{S} = \{s_1, \dots, s_m\}$ be the set of skills, and $\tau$ be the single incoming task. Define the state space

$$V = \mathcal{A} \,\cup\, \mathcal{S} \,\cup\, \{\tau\}, \qquad |V| = n + m + 1.$$

A *score vector* $\mathbf{r} \in \mathbb{R}^{|V|}$ assigns a real number to every node. After convergence, the entries $\mathbf{r}[a]$ for agents $a \in \mathcal{A}$ are the per-agent **HARP scores** that determine the ranking.

### 1.2 — The three transition matrices

Each matrix is **column-stochastic** ($\sum_i M_{ij} = 1$ for every $j$). Equivalently, $\mathbf{M}^\top \mathbf{1} = \mathbf{1}$.

| Matrix | Encodes | Non-zero block |
|---|---|---|
| $\mathbf{M}^P$ | performance | agent $\leftrightarrow$ skill, weighted by $W^P_{a,s}$ |
| $\mathbf{M}^C$ | endorsement | agent $\leftrightarrow$ agent, weighted by $W^C_{a,a'}$ |
| $\mathbf{M}^\phi$ | semantic | task $\leftrightarrow$ skill and task $\leftrightarrow$ agent, weighted by $\cos(\phi(\tau), \phi(\cdot))$ |

### 1.3 — Combined operator

$$\mathbf{M}_\tau \;=\; \beta_P \, \mathbf{M}^P \;+\; \beta_C \, \mathbf{M}^C \;+\; \beta_\phi \, \mathbf{M}^\phi, \qquad \beta_P + \beta_C + \beta_\phi = 1.$$

Because a **convex combination of column-stochastic matrices is column-stochastic**, $\mathbf{M}_\tau$ is a valid transition matrix.

### 1.4 — Softmax-temperature teleport

For each non-task node $v$, compute $\sigma_v = \cos(\phi(v), \phi(\tau))$. Then

$$p_\tau[v] \;=\; \frac{\exp(\kappa \, \sigma_v)}{\sum_{u \in V} \exp(\kappa \, \sigma_u)}, \qquad p_\tau[\tau] = 0.$$

High $\kappa$ → sharp focus on the most similar nodes. Low $\kappa$ → flatter prior.

### 1.5 — The update

$$\boxed{\;\mathbf{r}_{t+1} \;=\; \alpha \, \mathbf{M}_\tau \, \mathbf{r}_t \;+\; (1 - \alpha) \, \mathbf{p}_\tau\;}$$

Iterate until $\|\mathbf{r}_{t+1} - \mathbf{r}_t\|_1 < \varepsilon$.

### 1.6 — Convergence theorem (informal)

Define $T(\mathbf{r}) = \alpha \mathbf{M}_\tau \mathbf{r} + (1 - \alpha) \mathbf{p}_\tau$. Then for any $\mathbf{r}, \mathbf{r}'$ on the probability simplex,

$$\|T(\mathbf{r}) - T(\mathbf{r}')\|_1 \;=\; \alpha \, \|\mathbf{M}_\tau(\mathbf{r} - \mathbf{r}')\|_1 \;\leq\; \alpha \, \|\mathbf{r} - \mathbf{r}'\|_1.$$

So $T$ is an $\alpha$-contraction in $\ell_1$. By the Banach fixed-point theorem, $T$ has a **unique fixed point** $\mathbf{r}^\star$ and the iteration converges geometrically with rate $\alpha$. For $\alpha = 0.85$ and tolerance $10^{-7}$ the iteration needs at most $\lceil \log_{1/\alpha}(10^7) \rceil \approx 100$ steps.

---

That is the entire algorithm. The remaining sections of this notebook are about constructing the weight matrices $W^P$, $W^C$, and the embedding $\phi$ from real data, then evaluating the result.

## 2 — Install dependencies

Four libraries do the heavy lifting:

- **`sentence-transformers`** gives us a fast 384-dimensional embedder for text (the `all-MiniLM-L6-v2` model). This is the only "model" in the whole pipeline — everything else is pure NumPy linear algebra.
- **`networkx`** is used only for the vanilla-PageRank baseline. HARP itself does not need it.
- **`scikit-learn`** provides `cosine_similarity`, which is faster and more numerically stable than rolling our own.
- **`seaborn`** + **`matplotlib`** handle the diagnostic plots at the end.

The `-q` flag keeps Colab's install output quiet so the notebook stays readable. If you are running locally and already have these installed, this cell is a no-op.

In [ ]:
# =============================================================================
# Production install: pinned major versions for reproducibility.
# - Pinning the MAJOR version (e.g. numpy>=1.26,<3) keeps API guarantees
#   while still picking up bug fixes.
# - Safe to re-run; pip will skip already-installed packages.
# - In a local environment, drop the leading "!" and run from a terminal.
# - In Colab the older versions of some libraries are pre-installed; the
#   constraints below are deliberately compatible with Colab as of 2026.
# =============================================================================
!pip -q install \
    "numpy>=1.26,<3" \
    "pandas>=2.0,<3" \
    "scikit-learn>=1.3,<2" \
    "networkx>=3.0,<4" \
    "matplotlib>=3.7,<4" \
    "seaborn>=0.12,<1" \
    "sentence-transformers>=2.7,<6" \
    "tqdm>=4.65,<5" \
    "psutil>=5.9,<8"

## 3 — Imports, logging, and configuration

In a production system you do three things that a research script usually skips:

1. **Set up structured logging** with a single configured `logging.Logger`, instead of scattering `print()` calls everywhere. The default formatter prints level, timestamp, and module so logs are grep-able and tail-able. We deliberately keep `print()` for *user-facing* output (rankings, tables) and route diagnostics through the logger.
2. **Guard optional dependencies** so a missing library produces a clear warning rather than an obscure `ImportError` halfway through a long run. Here `seaborn` and `tqdm` are treated as optional — the notebook degrades gracefully if either is absent.
3. **Centralize configuration** in an immutable `HARPConfig` dataclass with `__post_init__` validation, instead of free-floating module globals. That way every function reads from a single source of truth and config errors fail at construction, not deep inside the power iteration.

The six HARP hyperparameters, in brief:

| Symbol | Default | What it does |
|---|---|---|
| $\alpha$ (`alpha`) | `0.85` | PageRank damping factor. Higher = walker stays on the graph longer and trusts the teleport prior less. |
| $\kappa$ (`kappa`) | `10.0` | Softmax temperature for the teleport vector. Higher = mass concentrates more sharply on the closest matches. |
| $\beta_P$ (`beta_p`) | `0.5` | Weight on the performance signal. |
| $\beta_C$ (`beta_c`) | `0.3` | Weight on the endorsement (co-success) signal. |
| $\beta_\phi$ (`beta_phi`) | `0.2` | Weight on the semantic signal. |
| `tolerance` | `1e-7` | $\ell_1$ convergence threshold for power iteration. |
| `max_iter` | `200` | Hard cap on iterations as a safety valve. |

The `__post_init__` validator enforces the **column-stochastic invariant**: $\beta_P + \beta_C + \beta_\phi = 1$ exactly. Without that constraint the combined transition matrix is no longer a Markov operator, and the algorithm loses its convergence guarantee. We fail fast at construction time rather than producing meaningless numbers downstream.

In [ ]:
from __future__ import annotations

import logging
import os
import random
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# ---- Required scientific stack -----------------------------------------------
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ---- Optional dependencies: degrade gracefully if missing --------------------
try:
    import seaborn as sns
    HAVE_SEABORN = True
except ImportError:
    sns = None
    HAVE_SEABORN = False

try:
    from tqdm.auto import tqdm
    HAVE_TQDM = True
except ImportError:
    HAVE_TQDM = False

    def tqdm(iterable=None, **kwargs):  # type: ignore[no-redef]
        """Fallback no-op tqdm so the notebook still runs without tqdm installed."""
        return iterable if iterable is not None else iter([])


# =============================================================================
# Structured logging (production hardening point #1)
# =============================================================================
# Use stdout-as-stream so the handler works in Colab, Jupyter, and headless CI.
# Set HARP_LOG_LEVEL=DEBUG in the environment to get more verbose output.
LOG_LEVEL = os.environ.get("HARP_LOG_LEVEL", "INFO").upper()
logging.basicConfig(
    level=LOG_LEVEL,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
    force=True,
)
logger = logging.getLogger("harp")


# =============================================================================
# Immutable, validated configuration (production hardening point #2)
# =============================================================================
@dataclass(frozen=True)
class HARPConfig:
    """
    Immutable, validated configuration for the HARP algorithm.

    Construction performs invariant checks via ``__post_init__`` so that any
    misconfiguration (e.g. non-convex betas, negative alpha) fails fast at
    object creation rather than producing meaningless rankings downstream.

    Attributes
    ----------
    alpha : float
        PageRank damping factor in (0, 1).
    kappa : float
        Softmax temperature for the teleport vector. Must be > 0.
    beta_p, beta_c, beta_phi : float
        Convex weights on the (performance, endorsement, semantic) matrices.
        Must each lie in [0, 1] and sum to exactly 1.0.
    tolerance : float
        L1-norm convergence threshold for power iteration.
    max_iter : int
        Safety cap on the number of power-iteration steps.
    seed : int
        Master seed for every PRNG used by the pipeline.
    artifact_dir : Path
        Directory to which embeddings and weight matrices are cached.

    Raises
    ------
    ValueError
        If any invariant is violated at construction time.
    """

    alpha: float = 0.85
    kappa: float = 10.0
    beta_p: float = 0.5
    beta_c: float = 0.3
    beta_phi: float = 0.2
    tolerance: float = 1e-7
    max_iter: int = 200
    seed: int = 42
    artifact_dir: Path = field(default_factory=lambda: Path(".harp_cache"))

    def __post_init__(self) -> None:
        if not (0.0 < self.alpha < 1.0):
            raise ValueError(f"alpha must be in (0, 1); got {self.alpha}")
        if self.kappa <= 0:
            raise ValueError(f"kappa must be > 0; got {self.kappa}")
        for name, beta in [("beta_p", self.beta_p),
                           ("beta_c", self.beta_c),
                           ("beta_phi", self.beta_phi)]:
            if not (0.0 <= beta <= 1.0):
                raise ValueError(f"{name} must be in [0, 1]; got {beta}")
        beta_sum = self.beta_p + self.beta_c + self.beta_phi
        if abs(beta_sum - 1.0) > 1e-9:
            raise ValueError(
                "beta_p + beta_c + beta_phi must equal 1.0 so that the "
                f"convex combination stays column-stochastic; got {beta_sum}"
            )
        if self.tolerance <= 0:
            raise ValueError(f"tolerance must be > 0; got {self.tolerance}")
        if self.max_iter < 1:
            raise ValueError(f"max_iter must be >= 1; got {self.max_iter}")

    @property
    def beta(self) -> Tuple[float, float, float]:
        return (self.beta_p, self.beta_c, self.beta_phi)


# Single canonical config object referenced everywhere below.
CONFIG = HARPConfig()

# Seed every random number generator we touch transitively.
np.random.seed(CONFIG.seed)
random.seed(CONFIG.seed)

# Ensure the artifact directory exists; safe to call repeatedly.
CONFIG.artifact_dir.mkdir(parents=True, exist_ok=True)

logger.info("HARP configuration loaded: %s", CONFIG)
logger.info(
    "Optional deps: seaborn=%s, tqdm=%s", HAVE_SEABORN, HAVE_TQDM
)

### 3a — Reproducibility & system-info report

A production run should record **exactly what environment produced the numbers**, so future-you (or your CI) can reproduce the result bit-for-bit (or at least know why it cannot). This cell dumps the Python version, the installed versions of every direct dependency, the operating system and architecture, and the available memory.

If you ever see a benchmark that "used to give NDCG@5 = 0.71 and now gives 0.68," the first thing to compare is the contents of this cell across runs. Nine times out of ten the culprit is a silent dependency upgrade (sentence-transformers shipping a new default model, scikit-learn changing its cosine-similarity dtype, etc.).

In [ ]:
import platform

try:
    import sklearn
    import sentence_transformers
except ImportError as exc:
    raise RuntimeError(
        "Required scientific stack not available. Re-run the install cell."
    ) from exc


def _report_environment() -> Dict[str, str]:
    """
    Collect a compact dictionary describing the runtime environment.

    Returns
    -------
    Dict[str, str]
        Mapping from environment-key to printable string value. Used both
        for logging and (optionally) for serialization next to result
        artifacts so reproducibility metadata travels with the numbers.
    """
    info: Dict[str, str] = {
        "python_version": sys.version.split()[0],
        "platform": f"{platform.system()} {platform.release()} ({platform.machine()})",
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__,
        "networkx": nx.__version__,
        "sentence-transformers": sentence_transformers.__version__,
        "seaborn": sns.__version__ if HAVE_SEABORN else "not installed",
    }

    # Memory is best-effort: psutil is in the install list but may be missing.
    try:
        import psutil
        mem = psutil.virtual_memory()
        info["total_memory_gb"] = f"{mem.total / (1024 ** 3):.1f}"
        info["available_memory_gb"] = f"{mem.available / (1024 ** 3):.1f}"
    except ImportError:
        info["total_memory_gb"] = "unknown (install psutil)"

    return info


_env = _report_environment()
logger.info("Runtime environment:")
for key, value in _env.items():
    logger.info("  %-22s %s", key + ":", value)

# Persist alongside future artifacts so result files always carry their
# environment metadata. JSON keeps it human-readable.
import json
(CONFIG.artifact_dir / "environment.json").write_text(json.dumps(_env, indent=2))
logger.info("Environment metadata written to %s", CONFIG.artifact_dir / "environment.json")

## 4 — The agent universe

In a production deployment this list would come from your agent registry — a database of every agent your platform exposes, with metadata like model family, scaffolds, and capabilities. For the tutorial we hand-craft **twelve agents** that reflect actual public scaffolds and model families seen on the HAL leaderboard for the GAIA benchmark.

Each agent gets two pieces of metadata:

1. A **name** (used in plots and as the unique identifier).
2. A short **natural-language description**. This is what the agent "advertises" itself as. The semantic signal will compare these strings to incoming task descriptions, so descriptions matter — they are part of the algorithm's input.

The descriptions are intentionally varied. Some agents brag about being **generalists** (good on broad tasks, but they will lose to specialists when the task is sharply scoped). Some are tightly **specialized** (the reverse trade-off). Watching how HARP balances these is half the fun of inspecting the rankings later.

In [ ]:
AGENT_NAMES: List[str] = [
    "HAL-Generalist-Sonnet-4.5",
    "HAL-Generalist-GPT-5",
    "HAL-Generalist-Gemini-2.5",
    "HF-OpenDeepResearch-GPT-4o",
    "HF-OpenDeepResearch-Llama3-70B",
    "AutoGen-Coder",
    "MetaGPT-Engineer",
    "CAMEL-Researcher",
    "Voyager-Skill-Agent",
    "GPTSwarm-Swarm",
    "WebArena-Browser",
    "SWE-Agent-Claude",
]

AGENT_DESC: Dict[str, str] = {
    "HAL-Generalist-Sonnet-4.5":
        "general purpose assistant with web browsing, code execution, "
        "and image understanding tools",
    "HAL-Generalist-GPT-5":
        "general purpose assistant with strong reasoning, code, "
        "and multi-step planning",
    "HAL-Generalist-Gemini-2.5":
        "general purpose assistant with multimodal vision and long-context "
        "reading",
    "HF-OpenDeepResearch-GPT-4o":
        "deep research agent that browses the web and reads PDF documents "
        "to answer questions",
    "HF-OpenDeepResearch-Llama3-70B":
        "open source deep research agent for web search and document reading",
    "AutoGen-Coder":
        "python code generation agent with iterative execution and debugging",
    "MetaGPT-Engineer":
        "software engineering multi-agent pipeline for writing and testing "
        "code",
    "CAMEL-Researcher":
        "role-playing research assistant for structured analysis and writing",
    "Voyager-Skill-Agent":
        "lifelong skill library agent that composes tools from past "
        "experience",
    "GPTSwarm-Swarm":
        "optimizable agent graph that combines several specialists in "
        "parallel",
    "WebArena-Browser":
        "specialist web browsing agent that navigates websites and clicks "
        "UI elements",
    "SWE-Agent-Claude":
        "software engineering agent for editing code repositories and "
        "writing patches",
}


def _validate_agent_registry(names: List[str], descriptions: Dict[str, str]) -> None:
    """
    Fail fast if the agent registry is internally inconsistent.

    Raises
    ------
    ValueError
        If names are duplicated, if any name lacks a description, or if any
        description is empty.
    """
    if len(set(names)) != len(names):
        raise ValueError("AGENT_NAMES contains duplicates.")
    missing = [n for n in names if n not in descriptions]
    if missing:
        raise ValueError(f"Missing AGENT_DESC entries for: {missing}")
    empty = [n for n, d in descriptions.items() if not d.strip()]
    if empty:
        raise ValueError(f"AGENT_DESC has empty descriptions for: {empty}")


_validate_agent_registry(AGENT_NAMES, AGENT_DESC)
n_agents = len(AGENT_NAMES)
logger.info("Agent registry validated: %d agents loaded.", n_agents)
for i, name in enumerate(AGENT_NAMES):
    logger.debug("  [%2d] %s", i, name)
print(f"Loaded {n_agents} agents:")
for i, name in enumerate(AGENT_NAMES):
    print(f"  [{i:2d}] {name}")

## 5 — The skill vocabulary

**Skills are the bridge between agents and tasks.** The GAIA benchmark's annotator metadata uses a free-form `Tools` field; we discretize this into eight skill categories that cover most of what GAIA L1/L2/L3 tasks demand. This discretization is what gives the performance matrix $W^P$ its column structure (one column per skill).

You may notice that some of the skills overlap semantically (`web_browsing` and `web_search` are close cousins; `pdf_reading` and `spreadsheet_analysis` both deal with structured documents). That overlap is **intentional** — it stress-tests whether HARP can still distinguish them, since their performance histories will differ even when their embeddings look similar.

In production you would expand this list, version-control it, and probably auto-derive new skills from clustering historical task descriptions. For the tutorial, eight is plenty.

In [ ]:
SKILLS: List[str] = [
    "web_browsing",
    "pdf_reading",
    "python_coding",
    "image_recognition",
    "math_calculation",
    "audio_transcription",
    "spreadsheet_analysis",
    "web_search",
]

SKILL_DESC: Dict[str, str] = {
    "web_browsing":
        "navigating interactive websites, clicking buttons, filling forms, "
        "and reading rendered pages",
    "pdf_reading":
        "extracting and understanding text and tables from PDF documents",
    "python_coding":
        "writing, executing, and debugging python programs to solve "
        "computational tasks",
    "image_recognition":
        "understanding the content of images, photographs, diagrams, and "
        "screenshots",
    "math_calculation":
        "performing arithmetic, algebra, and symbolic mathematics with "
        "precision",
    "audio_transcription":
        "transcribing spoken language from audio files into text",
    "spreadsheet_analysis":
        "reading excel and csv files and computing aggregations over rows "
        "and columns",
    "web_search":
        "issuing search engine queries and synthesizing information from "
        "results",
}


def _validate_skill_vocabulary(names: List[str], descriptions: Dict[str, str]) -> None:
    """Fail fast if the skill vocabulary is internally inconsistent."""
    if len(set(names)) != len(names):
        raise ValueError("SKILLS contains duplicates.")
    missing = [s for s in names if s not in descriptions]
    if missing:
        raise ValueError(f"Missing SKILL_DESC entries for: {missing}")
    empty = [s for s, d in descriptions.items() if not d.strip()]
    if empty:
        raise ValueError(f"SKILL_DESC has empty descriptions for: {empty}")


_validate_skill_vocabulary(SKILLS, SKILL_DESC)
n_skills = len(SKILLS)
logger.info("Skill vocabulary validated: %d skills loaded.", n_skills)
print(f"Loaded {n_skills} skills:")
for i, skill in enumerate(SKILLS):
    print(f"  [{i}] {skill:22s}  {SKILL_DESC[skill][:60]}...")

## 6 — Synthesizing GAIA-style tasks

In a production system you would stream tasks from a queue. For the tutorial we generate **thirty synthetic tasks** that mimic GAIA's question style and difficulty mix.

GAIA tasks are graded into three levels:

- **Level 1 (~31%)**: single-step lookup tasks. *"What is the population of Tokyo?"*
- **Level 2 (~53%)**: multi-step reasoning, often requiring a tool. *"Open this spreadsheet and return the sum of column B for entries in 2024."*
- **Level 3 (~16%)**: complex multi-modal tasks needing several tools chained together. *"Listen to this podcast clip, identify the speaker, and look up their previous employer."*

We sample levels with those probabilities and pair each task with a template plus a place/entity filler. The `level` column is preserved for downstream stratified analysis — e.g. testing whether HARP's gains over baselines are larger on L3 tasks than L1 (we predict yes, since L3 tasks have more skill diversity, which is exactly where the multi-relational graph adds value).

If you want to use **real GAIA** later, replace the synthesis block with `datasets.load_dataset("gaia-benchmark/GAIA", "2023_all", split="validation")` once you have access to the gated repository, and pull the `Question` field into the `desc` column. The rest of the pipeline does not care whether the tasks are real or synthetic.

In [ ]:
# Templates cover the eight skill categories with enough lexical diversity
# that the sentence embedder can map each one to its closest skill.
TASK_TEMPLATES: List[str] = [
    "Find the population of {place} from the official census PDF and compute "
    "its growth rate.",
    "Open this spreadsheet and return the sum of column B for entries in 2024.",
    "Transcribe the attached audio clip and identify the speaker.",
    "Browse the wikipedia page for {place} and extract the founding year.",
    "Run the attached python script and report the final output.",
    "Identify the country shown in this satellite image and look up its "
    "capital city.",
    "Search the web for the latest CEO of {place} and find their previous "
    "role.",
    "Open the attached PDF and find the date of the third figure caption.",
    "Compute the integral of the expression on page 4 of the document.",
    "Listen to the recording and write a five sentence summary in english.",
]

PLACE_FILLERS: List[str] = [
    "Paris", "Brazil", "Tokyo", "Apollo 11", "Marie Curie",
    "the Eiffel Tower", "OpenAI", "Anthropic", "Mumbai", "Iceland",
]

# GAIA validation-split level distribution (rounded). Encoded as a constant
# so a future swap to real GAIA only requires changing the loader, not the
# downstream sampling logic.
GAIA_LEVEL_MIX: Tuple[float, float, float] = (0.31, 0.53, 0.16)


def generate_tasks(n_tasks: int = 30, seed: int = CONFIG.seed) -> pd.DataFrame:
    """
    Generate synthetic GAIA-style tasks.

    Parameters
    ----------
    n_tasks : int
        Number of tasks to sample. Must be >= 1.
    seed : int
        Master seed for the per-call RNG. Defaults to the project-wide
        ``CONFIG.seed`` for full reproducibility.

    Returns
    -------
    pd.DataFrame
        Columns: ``task_id`` (str), ``desc`` (str), ``level`` (int in {1,2,3}).

    Raises
    ------
    ValueError
        If ``n_tasks`` is not a positive integer.
    """
    if n_tasks < 1:
        raise ValueError(f"n_tasks must be >= 1; got {n_tasks}")

    rng = np.random.default_rng(seed)
    levels = rng.choice([1, 2, 3], size=n_tasks, p=list(GAIA_LEVEL_MIX))
    rows = []
    for i, lvl in enumerate(levels):
        template = TASK_TEMPLATES[rng.integers(0, len(TASK_TEMPLATES))]
        place = PLACE_FILLERS[rng.integers(0, len(PLACE_FILLERS))]
        desc = template.format(place=place) if "{place}" in template else template
        rows.append({
            "task_id": f"task_{i:03d}",
            "desc": desc,
            "level": int(lvl),
        })
    return pd.DataFrame(rows)


TASKS = generate_tasks(n_tasks=30, seed=CONFIG.seed)
logger.info(
    "Generated %d tasks (L1=%d, L2=%d, L3=%d)",
    len(TASKS),
    (TASKS.level == 1).sum(),
    (TASKS.level == 2).sum(),
    (TASKS.level == 3).sum(),
)
print(f"Generated {len(TASKS)} tasks.")
print(f"  Level 1 (easy):   {(TASKS.level == 1).sum()}")
print(f"  Level 2 (medium): {(TASKS.level == 2).sum()}")
print(f"  Level 3 (hard):   {(TASKS.level == 3).sum()}")
print()
print("First five tasks:")
TASKS.head(5)

## 7 — Embedding agents, skills, and tasks (with disk cache)

Sentence embeddings are how we let HARP "read" the descriptions. The `all-MiniLM-L6-v2` model from `sentence-transformers` maps any sentence to a **384-dimensional unit vector** such that two sentences with similar meaning have a high cosine similarity (and vice versa).

We embed all three corpora *once* and reuse the vectors throughout the algorithm:

- `agent_emb` has shape `(n_agents, 384)` — one row per agent description.
- `skill_emb` has shape `(n_skills, 384)` — one row per skill description.
- `task_emb` has shape `(n_tasks, 384)` — one row per task description.

Because we pass `normalize_embeddings=True`, every row is a unit vector. That means **cosine similarity reduces to a plain dot product**, which lets all downstream calls to `cosine_similarity(...)` run as a single matrix multiply. This is the difference between embeddings being "fast enough" and being a bottleneck.

### Production hardening: disk-based caching

In a research notebook you embed every time the cell runs and it works fine. In a production loop (or a CI pipeline that runs this notebook on every commit) you want the embeddings cached on disk and reloaded from `.npy` files on subsequent runs — saves ~10 seconds per re-execution and removes a hidden dependency on the model-download endpoint.

The cache key includes a **content hash** of the input strings so that any edit to `AGENT_DESC`, `SKILL_DESC`, or `TASKS` automatically invalidates the cache. No silent staleness.

> **First-run note.** The embedder weights (~80 MB) are downloaded on first use into the HuggingFace cache. After that, our additional `.harp_cache/` folder makes re-runs effectively free.

In [ ]:
import hashlib
import time

EMBEDDER_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


def _content_hash(strings: List[str]) -> str:
    """Stable 12-hex-char SHA-256 prefix over a list of strings (used as cache key)."""
    h = hashlib.sha256()
    for s in strings:
        h.update(s.encode("utf-8"))
        h.update(b"\0")  # separator
    return h.hexdigest()[:12]


def embed_with_cache(
    strings: List[str],
    embedder: SentenceTransformer,
    cache_label: str,
    cache_dir: Path = CONFIG.artifact_dir,
) -> np.ndarray:
    """
    Encode ``strings`` with ``embedder`` and cache the result to disk.

    The cache key combines ``cache_label`` with a content hash of the
    input strings, so any edit to the inputs invalidates the cache
    automatically (no silent staleness).

    Parameters
    ----------
    strings : List[str]
        Sentences to embed. Must be non-empty.
    embedder : SentenceTransformer
        The loaded sentence-transformers model.
    cache_label : str
        Short human-readable tag included in the cache filename.
    cache_dir : Path
        Directory to write the cache file into.

    Returns
    -------
    np.ndarray of shape (len(strings), embedder_dim), L2-normalized rows.

    Raises
    ------
    ValueError
        If ``strings`` is empty.
    """
    if not strings:
        raise ValueError("Cannot embed an empty string list.")

    cache_key = _content_hash(strings)
    cache_path = cache_dir / f"emb_{cache_label}_{cache_key}.npy"

    if cache_path.exists():
        logger.info("Loading cached embeddings: %s", cache_path.name)
        return np.load(cache_path)

    logger.info("Cache miss for %s; encoding %d strings...", cache_label, len(strings))
    t0 = time.perf_counter()
    arr = embedder.encode(
        strings,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    elapsed = time.perf_counter() - t0
    np.save(cache_path, arr)
    logger.info(
        "Encoded %d strings in %.2fs and cached to %s",
        len(strings), elapsed, cache_path.name,
    )
    return arr


# Load embedder once (lazy: first call downloads ~80MB into HF cache).
logger.info("Loading sentence transformer (%s)...", EMBEDDER_MODEL_NAME)
embedder = SentenceTransformer(EMBEDDER_MODEL_NAME)

agent_emb = embed_with_cache(
    [AGENT_DESC[a] for a in AGENT_NAMES], embedder, cache_label="agents",
)
skill_emb = embed_with_cache(
    [SKILL_DESC[s] for s in SKILLS], embedder, cache_label="skills",
)
task_emb = embed_with_cache(
    TASKS["desc"].tolist(), embedder, cache_label="tasks",
)

print("Embeddings ready:")
print(f"  agent_emb: {agent_emb.shape}")
print(f"  skill_emb: {skill_emb.shape}")
print(f"  task_emb:  {task_emb.shape}")

# Quick sanity check: for every skill, which agent is closest?
print("\nSanity check — closest agent per skill (semantic neighbors):")
agent_skill_sim = cosine_similarity(skill_emb, agent_emb)
for si, skill in enumerate(SKILLS):
    best_ai = int(np.argmax(agent_skill_sim[si]))
    print(f"  {skill:22s} -> {AGENT_NAMES[best_ai]}")

## 8 — Simulating an outcome history

This is the most consequential synthetic step in the whole tutorial, so it gets its own paragraph of justification.

**Each agent has a hidden "true skill" vector** — its real probability of succeeding at each of the 8 skills. We draw these from a Beta$(2, 2)$ distribution so the true skills look like realistic empirical rates centered around 0.5, plus a small Gaussian jitter to break ties. The Beta distribution has support on $[0, 1]$ which is exactly the range a probability should occupy, and Beta$(2, 2)$ is the simplest non-uniform symmetric prior.

**Each task implicitly requires the 2 skills whose descriptions are semantically closest to it.** That definition lets us derive a per-task ground truth without manually labeling each task. It also keeps the evaluation honest — HARP does not know which two skills are "required" at test time; it has to infer them from the task description alone.

**The simulator runs the training period:** for each training task, we sample 4–6 attempters at random; each agent succeeds with probability equal to its mean true-skill on the task's required skills. We log every (agent, task, skill, outcome) tuple into the history dataframe $H$.

The point of this construction is to give HARP a **partial and noisy** history, which is what real systems have. No agent has tried every task, and even on tasks they have tried the outcomes are stochastic. We split the 30 tasks half-and-half into a **training set** (used to build $H$) and a **test set** (held out for evaluation).

In [ ]:
def simulate_history(
    agent_emb: np.ndarray,
    skill_emb: np.ndarray,
    task_emb: np.ndarray,
    train_idx: List[int],
    seed: int = CONFIG.seed,
    attempters_lo: int = 4,
    attempters_hi: int = 7,
) -> Tuple[pd.DataFrame, np.ndarray, List[List[int]]]:
    """
    Simulate a sparse outcome history H plus the ground-truth skill matrix.

    Each agent has a latent "true skill" vector drawn from Beta(2, 2) with
    Gaussian jitter. Each task implicitly requires the two skills whose
    embeddings are most semantically similar to it. For every training task,
    ``[attempters_lo, attempters_hi)`` random agents attempt it; each
    succeeds with probability equal to its mean true skill on the required
    skills.

    Parameters
    ----------
    agent_emb, skill_emb, task_emb : np.ndarray
        Embedding matrices from Section 7.
    train_idx : List[int]
        Indices into ``task_emb`` of tasks that should be observed in H.
    seed : int
        Master RNG seed.
    attempters_lo, attempters_hi : int
        Inclusive lower, exclusive upper bound on number of attempters per task.

    Returns
    -------
    H : pd.DataFrame
        Columns ``agent``, ``task``, ``skill``, ``outcome`` (all int).
    true_skill : np.ndarray of shape (n_agents, n_skills)
        Hidden ground-truth skill matrix.
    task_required_skills : List[List[int]]
        Per-task list of the two required skill indices.

    Raises
    ------
    ValueError
        If ``train_idx`` is empty or ``attempters_lo`` >= ``attempters_hi``.
    """
    if not train_idx:
        raise ValueError("train_idx must contain at least one task index.")
    if attempters_lo >= attempters_hi:
        raise ValueError("attempters_lo must be strictly less than attempters_hi.")
    n_a = len(AGENT_NAMES)
    n_s = len(SKILLS)
    if attempters_hi > n_a + 1:
        raise ValueError(
            f"Cannot sample {attempters_hi - 1} attempters from {n_a} agents."
        )

    rng = np.random.default_rng(seed)

    # Latent true-skill matrix: Beta(2,2) + small Gaussian jitter, clipped to (0, 1).
    true_skill = np.clip(
        rng.beta(2, 2, size=(n_a, n_s))
        + 0.15 * rng.standard_normal(size=(n_a, n_s)),
        0.02, 0.98,
    )

    # Each task's "required skills" = the top-2 most semantically similar.
    task_skill_sim = cosine_similarity(task_emb, skill_emb)
    task_required_skills = [np.argsort(-row)[:2].tolist() for row in task_skill_sim]

    # Simulate attempts (progress bar makes the wait less mysterious in long runs).
    history_rows: List[Dict[str, int]] = []
    iterator = tqdm(train_idx, desc="Simulating history", disable=not HAVE_TQDM)
    for ti in iterator:
        skills_i = task_required_skills[ti]
        n_attempters = int(rng.integers(attempters_lo, attempters_hi))
        attempters = rng.choice(n_a, size=n_attempters, replace=False)
        for ai in attempters:
            p_success = float(np.mean(true_skill[ai, skills_i]))
            outcome = int(rng.random() < p_success)
            for si in skills_i:
                history_rows.append({
                    "agent": int(ai),
                    "task": int(ti),
                    "skill": int(si),
                    "outcome": outcome,
                })

    return pd.DataFrame(history_rows), true_skill, task_required_skills


# Half-and-half split: tasks 0..14 build the history, 15..29 are held out.
n_train = len(TASKS) // 2
train_idx = list(range(n_train))
test_idx = list(range(n_train, len(TASKS)))

H, true_skill, task_required_skills = simulate_history(
    agent_emb, skill_emb, task_emb, train_idx, seed=CONFIG.seed,
)

logger.info(
    "History H built: %d rows | success rate=%.2f%% | train=%d test=%d",
    len(H), 100 * H.outcome.mean(), len(train_idx), len(test_idx),
)
print(f"History H has {len(H)} (agent, task, skill, outcome) tuples.")
print(f"Overall success rate in training: {H.outcome.mean():.2%}")
print(f"Train tasks: {len(train_idx)},  Test tasks: {len(test_idx)}")
print(f"true_skill shape: {true_skill.shape}")
print("\nFirst few history rows:")
H.head(8)

## 9 — Building the performance matrix $W^P$ (Bayesian smoothing)

Now we convert the raw history $H$ into a **performance weight per agent–skill pair**, stored in the matrix $W^P \in \mathbb{R}^{n \times m}$.

The naive choice would be the raw success rate $n^+ / (n^+ + n^-)$, but this has a famous problem: **an agent that tried one task and succeeded gets a perfect 1.0**, while an agent that tried 100 tasks and succeeded 99 times gets 0.99. A system using raw rates would prefer the lucky one-shot agent — exactly the wrong choice.

**Bayesian smoothing with a Beta$(1, 1)$ prior** fixes this cleanly. The posterior-mean estimate is

$$w^P_{a, s} \;=\; \frac{1 + n^+_{a, s}}{2 + n^+_{a, s} + n^-_{a, s}}.$$

This is the mean of the posterior $\theta \mid \text{data} \sim \text{Beta}(1 + n^+, 1 + n^-)$ where the prior $\text{Beta}(1, 1)$ is the uniform distribution on $[0, 1]$ (the principle-of-insufficient-reason choice). It has three properties we want:

1. With **zero observations** it returns $1 / 2 = 0.5$ — a principled "I don't know."
2. As observations accumulate it **converges to the empirical rate**.
3. It always lies strictly in $(0, 1)$, so multiplying by it never zeroes out a column.

This single function — `beta_smoothed` — is also our **cold-start solution for new agents**. Add a new agent with zero history and it shows up everywhere with a 0.5 entry, getting a fair shot at every task until evidence accumulates.

In [ ]:
def beta_smoothed(successes: int, attempts: int) -> float:
    """
    Posterior mean of the success probability under a Beta(1, 1) prior.

    Parameters
    ----------
    successes : int
        Observed number of successes. Must be >= 0.
    attempts : int
        Total observed attempts. Must be >= ``successes``.

    Returns
    -------
    float
        Posterior mean in (0, 1). With zero attempts returns 0.5.

    Raises
    ------
    ValueError
        If ``attempts`` < ``successes`` or either is negative.
    """
    if successes < 0 or attempts < 0:
        raise ValueError("successes and attempts must be non-negative.")
    if successes > attempts:
        raise ValueError(
            f"successes ({successes}) cannot exceed attempts ({attempts})."
        )
    failures = attempts - successes
    return (1 + successes) / (2 + successes + failures)


def build_performance_matrix(H: pd.DataFrame) -> np.ndarray:
    """
    Build the (n_agents x n_skills) performance matrix W^P from history H.

    Implementation note (production hardening): the previous version used a
    double Python loop with a DataFrame filter per cell, which is O(n_a * n_s)
    DataFrame scans. This version computes counts in a single pandas
    ``groupby().agg(...)`` and then does the Beta(1,1) smoothing as one
    vectorized array operation. ~50-100x faster on realistic-sized histories.

    Parameters
    ----------
    H : pd.DataFrame
        Must contain columns ``agent``, ``skill``, ``outcome``.

    Returns
    -------
    np.ndarray of shape (n_agents, n_skills), values strictly in (0, 1).

    Raises
    ------
    ValueError
        If H is missing the required columns.
    """
    required = {"agent", "skill", "outcome"}
    missing = required - set(H.columns)
    if missing:
        raise ValueError(f"History H is missing required columns: {missing}")

    n_a = len(AGENT_NAMES)
    n_s = len(SKILLS)

    # One groupby instead of n_a * n_s filters.
    agg = (
        H.groupby(["agent", "skill"], sort=False)
         .agg(n_succ=("outcome", "sum"), n_att=("outcome", "count"))
         .reset_index()
    )

    succ = np.zeros((n_a, n_s), dtype=np.int64)
    att = np.zeros((n_a, n_s), dtype=np.int64)
    succ[agg.agent.to_numpy(), agg.skill.to_numpy()] = agg.n_succ.to_numpy()
    att[agg.agent.to_numpy(), agg.skill.to_numpy()] = agg.n_att.to_numpy()

    # Vectorized Beta(1, 1) smoothing.
    return (1 + succ) / (2 + att)


W_P = build_performance_matrix(H)

# Runtime invariant: W^P must lie strictly in (0, 1) by Beta-smoothing.
assert np.all((W_P > 0) & (W_P < 1)), "W_P entries must be strictly in (0, 1)"

logger.info(
    "W_P computed: shape=%s min=%.3f max=%.3f mean=%.3f",
    W_P.shape, W_P.min(), W_P.max(), W_P.mean(),
)
print("Performance matrix W^P computed.")
print(f"  shape: {W_P.shape}")
print(f"  min:   {W_P.min():.3f}  (worst agent on hardest skill)")
print(f"  max:   {W_P.max():.3f}  (best agent on easiest skill)")
print(f"  mean:  {W_P.mean():.3f}  (should be near 0.5 for sparse history)")

print("\nW^P (rows = agents, cols = skills):")
W_P_df = pd.DataFrame(W_P, index=AGENT_NAMES, columns=SKILLS).round(2)
W_P_df

## 10 — Building the endorsement matrix $W^C$ (co-success / co-failure)

Endorsements model the soft statement *"agent $a$ trusts agent $a'$ enough to hand it work, and that work turned out well."*

In GAIA traces we do not have **direct invocation logs** (no agent literally hands off work to another agent), so we approximate the endorsement signal via **co-success**:

- If agents $i$ and $j$ both **succeed** on the same training task, increment $W^C_{i,j}$ by $+1.0$.
- If they both **fail** on the same task, decrement $W^C_{i,j}$ by $-0.3$ and clip at zero.

The asymmetric reward/penalty (1.0 vs 0.3) is deliberate: succeeding together is strong evidence of complementary competence, while failing together is weak evidence of incompetence (the task itself may just have been hard). Clipping at zero ensures we never have negative entries, which would break the column-stochastic property of the eventual transition matrix.

This is conceptually the same construction **EigenTrust** (Kamvar, Schlosser, Garcia-Molina 2003) uses for peer-to-peer reputation, just applied to LLM agents instead of file-sharing peers.

> If you ever deploy HARP with **real invocation logs** (agent A explicitly delegated subtask X to agent B, and B succeeded), replace this co-success construction with the actual handoff counts — the signal will get dramatically sharper.

In [ ]:
def build_endorsement_matrix(
    H: pd.DataFrame,
    train_idx: List[int],
    co_success_reward: float = 1.0,
    co_failure_penalty: float = 0.3,
) -> np.ndarray:
    """
    Build the (n_agents x n_agents) endorsement matrix W^C via co-success
    (positive reward) and co-failure (penalty, clipped at zero).

    Implementation note (production hardening): the previous version used a
    Python quadruple nested loop O(|train_idx| * n_a^2). This version
    computes the symmetric outer products per task using NumPy broadcasting
    in a single vectorized pass.

    Parameters
    ----------
    H : pd.DataFrame
        Must contain columns ``agent``, ``task``, ``outcome``.
    train_idx : List[int]
        Task indices to include in the construction.
    co_success_reward : float
        Increment added to W^C[i, j] when agents i, j both succeed.
    co_failure_penalty : float
        Decrement subtracted (clipped at 0) when both fail.

    Returns
    -------
    np.ndarray of shape (n_agents, n_agents), non-negative, zero diagonal.

    Raises
    ------
    ValueError
        If H lacks required columns or the constants are not non-negative.
    """
    required = {"agent", "task", "outcome"}
    missing = required - set(H.columns)
    if missing:
        raise ValueError(f"History H is missing required columns: {missing}")
    if co_success_reward < 0 or co_failure_penalty < 0:
        raise ValueError("reward and penalty must be non-negative.")

    n_a = len(AGENT_NAMES)
    W_C = np.zeros((n_a, n_a), dtype=float)
    H_train = H[H.task.isin(train_idx)]

    # Group once; iterate per task.
    for ti, df_t in H_train.groupby("task"):
        # Indicator vectors over agents for success / failure on this task.
        succ_vec = np.zeros(n_a, dtype=float)
        fail_vec = np.zeros(n_a, dtype=float)
        succ_agents = df_t.loc[df_t.outcome == 1, "agent"].unique()
        fail_agents = df_t.loc[df_t.outcome == 0, "agent"].unique()
        succ_vec[succ_agents] = 1.0
        fail_vec[fail_agents] = 1.0

        # Outer-product co-occurrence with zero diagonal.
        succ_pairs = np.outer(succ_vec, succ_vec)
        np.fill_diagonal(succ_pairs, 0.0)
        W_C += co_success_reward * succ_pairs

        fail_pairs = np.outer(fail_vec, fail_vec)
        np.fill_diagonal(fail_pairs, 0.0)
        W_C = np.maximum(W_C - co_failure_penalty * fail_pairs, 0.0)

    return W_C


W_C = build_endorsement_matrix(H, train_idx)

# Runtime invariants for W^C.
assert W_C.shape == (n_agents, n_agents), "W_C must be square over agents"
assert np.all(W_C >= 0), "W_C must be non-negative after clipping"
assert np.all(np.diag(W_C) == 0), "W_C diagonal must be zero (no self-endorsements)"

logger.info(
    "W_C computed: shape=%s density=%.2f%% most-endorsed=%s",
    W_C.shape, 100 * (W_C > 0).mean(),
    AGENT_NAMES[int(W_C.sum(axis=0).argmax())],
)
print("Endorsement matrix W^C computed.")
print(f"  shape: {W_C.shape}")
print(f"  density (non-zero entries): {(W_C > 0).mean():.2%}")
print(f"  most-endorsed agent: {AGENT_NAMES[int(W_C.sum(axis=0).argmax())]}")
print(f"  most-endorsing agent: {AGENT_NAMES[int(W_C.sum(axis=1).argmax())]}")

print("\nW^C (square matrix, agents x agents):")
W_C_df = pd.DataFrame(W_C, index=AGENT_NAMES, columns=AGENT_NAMES).astype(int)
W_C_df

## 11 — Column normalization helper

PageRank — and HARP — require **column-stochastic** transition matrices: every column sums to one, so that each column represents a probability distribution over where the random walker goes *from* that node.

We will need this normalization several times in the next section, so we wrap it in a small helper. Two safety features matter:

1. **Avoid division by zero** using a tiny `eps`. A column whose total is essentially zero would otherwise produce `NaN`s that silently propagate through the whole matrix.
2. **Leave all-zero columns untouched** (`col_sums = 1.0` in that case), so the caller can detect dangling nodes and apply a uniform-distribution fix afterwards. This is the standard treatment of "dead-end" nodes that Brin & Page (1998) introduced for PageRank.

The dangling-node fix is what guarantees the resulting Markov chain is **ergodic** — every state is reachable from every other state with positive probability — which is a prerequisite for the chain to have a unique stationary distribution.

In [ ]:
def col_normalize(W: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    Normalize columns of W to sum to 1.

    Columns whose total is essentially zero are returned untouched (column
    sum left as 1 by substitution), so the caller can detect dangling
    nodes and apply a uniform-distribution fix afterwards.
    """
    col_sums = W.sum(axis=0, keepdims=True)
    col_sums = np.where(col_sums < eps, 1.0, col_sums)
    return W / col_sums


# Sanity check — must hold by construction.
_test = col_normalize(np.array([[1.0, 2.0], [3.0, 4.0]]))
assert np.allclose(_test.sum(axis=0), 1.0)
print("col_normalize helper ready, sanity check passed.")

## 12 — Building the three task-conditional transition matrices

**This is the heart of HARP.** Everything before this section was preparation; everything after is evaluation. Read this section twice.

### 12.1 — The state space, indexed concretely

We lay out the state vector $\mathbf{r}$ over

$$V = \mathcal{A} \cup \mathcal{S} \cup \{\tau\}, \qquad |V| = n_{\text{agents}} + n_{\text{skills}} + 1 = 12 + 8 + 1 = 21.$$

We use a concrete index layout:

| Index range | Node group |
|---|---|
| `0 .. n_agents-1` | agents (12 indices) |
| `n_agents .. n_agents + n_skills - 1` | skills (8 indices) |
| `n_agents + n_skills` | the single task node |

The three index arrays `A_idx`, `S_idx`, `T_idx` make the block structure of every matrix readable.

### 12.2 — The performance matrix $\mathbf{M}^P$

Probability flows back and forth between agents and skills, weighted by $W^P$.

- **Skill → agent**: column-normalize $W^P{}^\top$. The column "skill $s$" then gives the probability of jumping to each agent, with agents that perform $s$ well getting more mass.
- **Agent → skill**: column-normalize $W^P$. Symmetric construction.

The task node is a **sink** in $\mathbf{M}^P$ (the performance signal has nothing to say about a task by itself; that is what the semantic signal $\mathbf{M}^\phi$ is for). We set `M_P[T_idx, T_idx] = 1.0` so the task column is a valid distribution.

### 12.3 — The endorsement matrix $\mathbf{M}^C$

Probability flows only among agents, weighted by $W^C$. The non-agent nodes are left with identity self-loops (`M_C = np.eye(V)` initialized), then we overwrite the agent submatrix with `col_normalize(W_C)`.

### 12.4 — The semantic matrix $\mathbf{M}^\phi$

The only signal where the **task node participates non-trivially**.

- **Task → skills**: half the outgoing mass is distributed to skills proportional to (rescaled) cosine similarity with the task.
- **Task → agents**: the other half goes directly to agents proportional to (rescaled) cosine similarity with the task.
- **Skills → task**: every skill sends mass back to the task proportional to its similarity. This is what closes the loop and lets credit flow `skill → task → skill → agent`.

We rescale cosines from $[-1, 1]$ to $[0, 1]$ by `(1 + cos) / 2` so we never get negative weights (negative weights would break the column-stochastic property).

### 12.5 — The dangling-node fix

Any column whose sum is essentially zero (a "dangling node" with no outgoing edges) gets replaced by the **uniform distribution** $\mathbf{1} / V$. This is the exact trick Brin & Page used in the original PageRank paper to make the chain ergodic.

### 12.6 — The contract

At the end we **assert** that all three matrices are column-stochastic. If any column does not sum to one, the random walker would be leaking probability mass and the iteration would not converge to a unique fixed point. **This is the mathematical guarantee in code form.**

In [ ]:
def make_transition_matrices(
    task_idx: int,
    W_P: np.ndarray,
    W_C: np.ndarray,
    agent_emb: np.ndarray,
    skill_emb: np.ndarray,
    task_emb: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, int]:
    """
    Build the three task-conditional column-stochastic transition matrices
    over the state space V = agents ∪ skills ∪ {task}.

    Implementation note (production hardening): the previous version had
    Python double loops to populate M^P and a per-row loop for M^C. This
    version uses NumPy slice assignment via ``np.ix_`` so the entire block
    is filled in one C-level memcpy. ~30x faster.

    Parameters
    ----------
    task_idx : int
        Index of the task node in ``task_emb``. Must satisfy 0 <= task_idx < len(task_emb).
    W_P : (n_agents, n_skills) ndarray
    W_C : (n_agents, n_agents) ndarray
    agent_emb, skill_emb, task_emb : ndarray
        Embedding matrices from Section 7.

    Returns
    -------
    M_P, M_C, M_phi : (V, V) ndarray, each column-stochastic
    A_idx, S_idx   : index arrays for agents and skills
    T_idx          : the single integer index of the task node

    Raises
    ------
    ValueError
        If task_idx is out of range or matrix shapes are inconsistent.
    """
    n_a = len(AGENT_NAMES)
    n_s = len(SKILLS)

    if not (0 <= task_idx < len(task_emb)):
        raise ValueError(
            f"task_idx={task_idx} out of range [0, {len(task_emb)})."
        )
    if W_P.shape != (n_a, n_s):
        raise ValueError(f"W_P shape mismatch: expected ({n_a}, {n_s}), got {W_P.shape}")
    if W_C.shape != (n_a, n_a):
        raise ValueError(f"W_C shape mismatch: expected ({n_a}, {n_a}), got {W_C.shape}")

    V = n_a + n_s + 1
    A_idx = np.arange(0, n_a)
    S_idx = np.arange(n_a, n_a + n_s)
    T_idx = n_a + n_s

    # ============================================================
    # M^P: Agent <-> Skill via performance (vectorized block fill)
    # ============================================================
    M_P = np.zeros((V, V))
    P_skill_to_agent = col_normalize(W_P.T)   # shape (n_skills, n_agents)
    P_agent_to_skill = col_normalize(W_P)     # shape (n_agents, n_skills)
    # M_P[agent rows, skill cols] = transitions FROM skill TO agent
    M_P[np.ix_(A_idx, S_idx)] = P_skill_to_agent.T   # (n_a, n_s)
    # M_P[skill rows, agent cols] = transitions FROM agent TO skill
    M_P[np.ix_(S_idx, A_idx)] = P_agent_to_skill.T   # (n_s, n_a)
    # Task is a sink in M^P (performance signal says nothing about the task).
    M_P[T_idx, T_idx] = 1.0
    # Dangling-node fix: any all-zero column -> uniform distribution.
    zero_cols = M_P.sum(axis=0) < 1e-12
    M_P[:, zero_cols] = 1.0 / V

    # ============================================================
    # M^C: Agent <-> Agent via endorsement (vectorized block fill)
    # ============================================================
    M_C = np.eye(V)                                  # identity self-loops everywhere
    C_agent = col_normalize(W_C)                     # (n_a, n_a) column-stochastic
    # Overwrite the entire agent-block in one operation.
    M_C[np.ix_(A_idx, A_idx)] = C_agent
    zero_cols = M_C.sum(axis=0) < 1e-12
    M_C[:, zero_cols] = 1.0 / V

    # ============================================================
    # M^phi: semantic edges to/from the task node
    # ============================================================
    M_phi = np.zeros((V, V))
    # Rescale cosines from [-1, 1] to [0, 1] to avoid negative weights.
    sim_skill_task = (1 + cosine_similarity(
        skill_emb, task_emb[task_idx:task_idx + 1]).ravel()) / 2
    sim_agent_task = (1 + cosine_similarity(
        agent_emb, task_emb[task_idx:task_idx + 1]).ravel()) / 2
    PHI_to_skills = sim_skill_task / max(sim_skill_task.sum(), 1e-12)
    PHI_to_agents = sim_agent_task / max(sim_agent_task.sum(), 1e-12)
    # FROM task -> skills (half) and -> agents (other half)
    M_phi[S_idx, T_idx] = 0.5 * PHI_to_skills
    M_phi[A_idx, T_idx] = 0.5 * PHI_to_agents
    # FROM skills -> task (back, weighted by similarity)
    M_phi[T_idx, S_idx] = sim_skill_task / (sim_skill_task.sum() + 1e-12)
    zero_cols = M_phi.sum(axis=0) < 1e-12
    M_phi[:, zero_cols] = 1.0 / V

    return M_P, M_C, M_phi, A_idx, S_idx, T_idx


# Quick verification — every column of every matrix MUST sum to 1.
_M_P, _M_C, _M_phi, _A_idx, _S_idx, _T_idx = make_transition_matrices(
    test_idx[0], W_P, W_C, agent_emb, skill_emb, task_emb,
)
assert np.allclose(_M_P.sum(axis=0), 1.0), "M_P columns must sum to 1"
assert np.allclose(_M_C.sum(axis=0), 1.0), "M_C columns must sum to 1"
assert np.allclose(_M_phi.sum(axis=0), 1.0), "M_phi columns must sum to 1"
logger.info(
    "State space V = %d (agents=%d + skills=%d + task=1); all matrices column-stochastic.",
    _M_P.shape[0], n_agents, n_skills,
)
print(f"State space: V = {_M_P.shape[0]} nodes (agents + skills + task).")
print(f"  A_idx = {list(_A_idx)}")
print(f"  S_idx = {list(_S_idx)}")
print(f"  T_idx = {_T_idx}")
print("All three transition matrices are column-stochastic. ✓")

## 13 — The task-conditional teleport vector $\mathbf{p}_\tau$

PageRank's **teleport vector** is what makes the random walker eventually visit every page even if the graph has dead ends — at each step, with probability $1 - \alpha$, the walker resets to a random node sampled from $\mathbf{p}$.

The history of the teleport vector tracks the evolution of personalized search:

- **Original PageRank** (Brin & Page 1998) used a **uniform** teleport: $\mathbf{p} = \mathbf{1} / V$. No personalization.
- **Personalized PageRank** (Haveliwala 2002) used a **delta** teleport on a single seed node — useful when you want pages "topically close to a single page."
- **Topic-Sensitive PageRank** pre-computed sixteen teleport vectors, one per high-level topic, and switched at query time.

**HARP generalizes all three.** Instead of a discrete topic, we have a continuous task description. We build a teleport vector by computing the cosine similarity between the task and every other node (agent or skill), then passing those similarities through a **softmax with temperature $\kappa$**:

$$p_\tau[v] \;=\; \frac{\exp(\kappa \, \sigma_v)}{\sum_{u} \exp(\kappa \, \sigma_u)}, \qquad p_\tau[\tau] = 0.$$

The task node itself receives **zero mass** so the walker is always pushed back into the agent/skill subgraph.

**Tuning $\kappa$:** a high $\kappa$ makes the teleport sharply concentrated on the most similar agents and skills (good when you trust the embedding). A low $\kappa$ spreads probability more evenly (good when the embedding is unreliable, or when you want HARP to explore more). The default $\kappa = 10$ is roughly the sweet spot empirically.

In [ ]:
def teleport_vector(
    task_idx: int,
    agent_emb: np.ndarray,
    skill_emb: np.ndarray,
    task_emb: np.ndarray,
    kappa: float = CONFIG.kappa,
) -> np.ndarray:
    """
    Softmax-temperature semantic teleport vector p_tau over V.

    Mass concentrates on agents and skills whose descriptions are most
    semantically similar to the current task. The task node itself
    receives zero mass.

    Numerical-stability note: the standard `exp(kappa * sims)` formulation
    overflows when `kappa * max(sims)` exceeds ~700 (double-precision
    range). We subtract `max(kappa * sims)` before the exp so the largest
    pre-softmax value is 0 — the standard numerically-stable softmax trick.

    Parameters
    ----------
    task_idx : int
        Index of the task in ``task_emb``.
    agent_emb, skill_emb, task_emb : ndarray
        Embedding matrices.
    kappa : float
        Softmax temperature, must be > 0.

    Returns
    -------
    np.ndarray of shape (V,) summing to 1.0 with task-node entry == 0.

    Raises
    ------
    ValueError
        If kappa <= 0 or task_idx is out of range.
    """
    if kappa <= 0:
        raise ValueError(f"kappa must be > 0; got {kappa}")
    if not (0 <= task_idx < len(task_emb)):
        raise ValueError(
            f"task_idx={task_idx} out of range [0, {len(task_emb)})."
        )

    n_a = len(AGENT_NAMES)
    n_s = len(SKILLS)
    sim_agents = cosine_similarity(
        agent_emb, task_emb[task_idx:task_idx + 1]).ravel()
    sim_skills = cosine_similarity(
        skill_emb, task_emb[task_idx:task_idx + 1]).ravel()
    sims = np.concatenate([sim_agents, sim_skills, np.array([0.0])])

    # Numerically stable softmax: subtract max before exp.
    logits = kappa * sims
    logits -= logits.max()
    p = np.exp(logits)
    p /= p.sum()
    return p


# Sanity check on the first held-out task.
_p = teleport_vector(test_idx[0], agent_emb, skill_emb, task_emb)
assert abs(_p.sum() - 1.0) < 1e-9, "teleport must be a probability distribution"
top_3_agents = np.argsort(-_p[:n_agents])[:3]

print(f"For task: \"{TASKS.iloc[test_idx[0]]['desc'][:70]}...\"")
print("Top 3 agents by semantic teleport mass (Sim-only baseline):")
for rank, ai in enumerate(top_3_agents, 1):
    print(f"  #{rank}: {AGENT_NAMES[ai]:35s}  mass={_p[ai]:.4f}")
print(f"\np_tau sum = {_p.sum():.6f}, task node mass = {_p[-1]:.6f}")

## 14 — The HARP power iteration

Now we **tie everything together** and write the main algorithm. The update rule is the single line

$$\mathbf{r}_{t+1} \;=\; \alpha \, \mathbf{M}_\tau \, \mathbf{r}_t \;+\; (1 - \alpha) \, \mathbf{p}_\tau, \qquad \text{where} \quad \mathbf{M}_\tau \;=\; \beta_P \mathbf{M}^P + \beta_C \mathbf{M}^C + \beta_\phi \mathbf{M}^\phi.$$

We initialize $\mathbf{r}_0 = \mathbf{p}_\tau$ (a sensible warm start — the teleport itself is already a reasonable first guess), and iterate until consecutive states differ by less than the tolerance in $\ell_1$ norm.

**Why convex combination keeps $\mathbf{M}_\tau$ column-stochastic.** If each summand has columns summing to one, and the weights $\beta$ sum to one, then

$$\sum_i \big(\beta_P M^P_{ij} + \beta_C M^C_{ij} + \beta_\phi M^\phi_{ij}\big) \;=\; \beta_P \cdot 1 + \beta_C \cdot 1 + \beta_\phi \cdot 1 \;=\; 1.$$

So $\mathbf{M}_\tau$ is column-stochastic. The personalized-PageRank update preserves probability mass (sum of $\mathbf{r}$ stays 1 forever).

**Convergence rate.** The update map $T(\mathbf{r}) = \alpha \mathbf{M}_\tau \mathbf{r} + (1 - \alpha) \mathbf{p}_\tau$ is an $\alpha$-contraction in $\ell_1$. So after $k$ iterations the error is at most $\alpha^k$ times the initial error. With $\alpha = 0.85$ and tolerance $10^{-7}$ we expect convergence in about

$$\big\lceil \log_{1/\alpha}(10^7) \big\rceil \;\approx\; \big\lceil 7 / \log_{10}(1/0.85) \big\rceil \;\approx\; 100 \;\text{iterations}.$$

Empirically the convergence is **much faster** than this worst case — usually 20–40 iterations — because real signal matrices are far from worst-case adversarial.

In [ ]:
class HARPConvergenceWarning(UserWarning):
    """Raised (as a warning) when HARP hits ``max_iter`` without converging."""


def harp_rank(
    task_idx: int,
    W_P: np.ndarray,
    W_C: np.ndarray,
    agent_emb: np.ndarray,
    skill_emb: np.ndarray,
    task_emb: np.ndarray,
    beta: Optional[Tuple[float, float, float]] = None,
    alpha: Optional[float] = None,
    kappa: Optional[float] = None,
    max_iter: Optional[int] = None,
    tol: Optional[float] = None,
    return_diagnostics: bool = False,
    warn_on_max_iter: bool = True,
):
    """
    Run the HARP algorithm for a single task and return per-agent scores.

    Update rule (Banach-contracting in L1 with rate alpha):

        r_{t+1} = alpha * M_tau * r_t + (1 - alpha) * p_tau

    where:
        M_tau = beta[0] * M^P + beta[1] * M^C + beta[2] * M^phi
        p_tau = softmax(kappa * cos(phi(task), phi(node)))

    Parameters
    ----------
    task_idx : int
        Index of the task in ``task_emb``.
    W_P, W_C : np.ndarray
        Weight matrices from Sections 9 and 10.
    agent_emb, skill_emb, task_emb : np.ndarray
        Embedding matrices from Section 7.
    beta, alpha, kappa, max_iter, tol : optional
        If None, fall back to the corresponding ``CONFIG`` value. Pass to
        override per-call for ablations / hyperparameter sweeps.
    return_diagnostics : bool
        If True, also return iteration count and the residual history.
    warn_on_max_iter : bool
        Emit a logger warning if convergence is not achieved within max_iter.

    Returns
    -------
    agent_scores : np.ndarray of shape (n_agents,)
    n_iters      : int, number of iterations until convergence
    residuals    : list[float] (only if return_diagnostics=True)

    Raises
    ------
    ValueError
        If beta is supplied but does not sum to 1, or alpha not in (0, 1).
    """
    # Resolve None args against CONFIG so callers can override surgically.
    beta = beta if beta is not None else CONFIG.beta
    alpha = alpha if alpha is not None else CONFIG.alpha
    kappa = kappa if kappa is not None else CONFIG.kappa
    max_iter = max_iter if max_iter is not None else CONFIG.max_iter
    tol = tol if tol is not None else CONFIG.tolerance

    # Per-call invariant checks (don't trust callers).
    if abs(sum(beta) - 1.0) > 1e-9:
        raise ValueError(f"beta must sum to 1; got {beta} summing to {sum(beta)}")
    if not (0.0 < alpha < 1.0):
        raise ValueError(f"alpha must be in (0, 1); got {alpha}")

    # Build the three column-stochastic transition matrices for this task.
    M_P, M_C, M_phi, A_idx, _, _ = make_transition_matrices(
        task_idx, W_P, W_C, agent_emb, skill_emb, task_emb,
    )

    # Convex combination -> still column-stochastic.
    M = beta[0] * M_P + beta[1] * M_C + beta[2] * M_phi

    # Semantic teleport vector.
    p = teleport_vector(task_idx, agent_emb, skill_emb, task_emb, kappa=kappa)

    # Power iteration.
    r = p.copy()
    residuals: List[float] = []
    converged = False
    for it in range(max_iter):
        r_new = alpha * (M @ r) + (1 - alpha) * p
        diff = float(np.linalg.norm(r_new - r, ord=1))
        residuals.append(diff)
        r = r_new
        if diff < tol:
            converged = True
            break

    if not converged and warn_on_max_iter:
        logger.warning(
            "HARP did not converge for task_idx=%d in max_iter=%d (last residual %.2e > tol %.2e)",
            task_idx, max_iter, residuals[-1], tol,
        )

    agent_scores = r[A_idx]
    if return_diagnostics:
        return agent_scores, it + 1, residuals
    return agent_scores, it + 1


# Try it on the first held-out task.
_scores, _n_iters = harp_rank(
    test_idx[0], W_P, W_C, agent_emb, skill_emb, task_emb,
)
print(f"Task: \"{TASKS.iloc[test_idx[0]]['desc']}\"")
print(f"Converged in {_n_iters} iterations.")
print("\nHARP top 5 agents:")
for rank, ai in enumerate(np.argsort(-_scores)[:5], 1):
    print(f"  #{rank}: {AGENT_NAMES[ai]:35s}  score={_scores[ai]:.4f}")

## 15 — Baselines for comparison

To know whether HARP is doing **anything useful**, we compare it against five reasonable alternatives. Each baseline ignores some of the signals HARP uses; together they reveal how much each component is worth.

| Baseline | Uses | Ignores |
|---|---|---|
| **Random** | nothing | everything — establishes a floor |
| **Perf-only** | $W^P$ averaged over skills | the task, the endorsement graph |
| **Sim-only** | semantic similarity to task | history, endorsements |
| **VanillaPR** | $W^C$ via classical PageRank | history, semantics |
| **WeightedAvg** | all three, linearly | the **graph structure** |

The closest competitor is **WeightedAvg**: it uses the same three signals as HARP, just combined linearly *after* min-max normalization, without the graph structure. The gap between HARP and WeightedAvg is the cleanest empirical measure of *"does the random-walk fusion actually help, beyond just averaging?"*

In [ ]:
def baseline_random(task_idx: int, **kwargs) -> np.ndarray:
    """Pure noise — establishes a performance floor."""
    return np.random.default_rng(task_idx).random(len(AGENT_NAMES))


def baseline_perf_only(task_idx: int, *, W_P: np.ndarray, **kwargs) -> np.ndarray:
    """Global mean performance across all skills — ignores the task entirely."""
    return W_P.mean(axis=1)


def baseline_sim_only(
    task_idx: int, *, agent_emb: np.ndarray, task_emb: np.ndarray, **kwargs
) -> np.ndarray:
    """Cosine similarity between agent description and task — ignores history."""
    return cosine_similarity(agent_emb, task_emb[task_idx:task_idx + 1]).ravel()


def baseline_vanilla_pagerank(
    task_idx: int, *, W_C: np.ndarray, **kwargs
) -> np.ndarray:
    """Classical PageRank on the endorsement graph with uniform teleport."""
    n_a = len(AGENT_NAMES)
    M = col_normalize(W_C)
    G = nx.from_numpy_array(M.T, create_using=nx.DiGraph)
    pr = nx.pagerank(G, alpha=0.85)
    return np.array([pr.get(i, 1.0 / n_a) for i in range(n_a)])


def baseline_weighted_avg(
    task_idx: int,
    *,
    W_P: np.ndarray,
    W_C: np.ndarray,
    agent_emb: np.ndarray,
    task_emb: np.ndarray,
    w: Tuple[float, float, float] = (0.4, 0.3, 0.3),
    **kwargs,
) -> np.ndarray:
    """Linear combination of perf + sim + PageRank after min-max normalization."""
    def normalize(x):
        return (x - x.min()) / (x.max() - x.min() + 1e-9)

    a = baseline_perf_only(task_idx, W_P=W_P)
    b = baseline_sim_only(task_idx, agent_emb=agent_emb, task_emb=task_emb)
    c = baseline_vanilla_pagerank(task_idx, W_C=W_C)
    return w[0] * normalize(a) + w[1] * normalize(b) + w[2] * normalize(c)


def baseline_harp(task_idx: int, **kwargs) -> np.ndarray:
    """HARP wrapper that matches the calling convention of the others."""
    scores, _ = harp_rank(task_idx, W_P, W_C, agent_emb, skill_emb, task_emb)
    return scores


print("All baselines defined: Random, Perf-only, Sim-only, VanillaPR, WeightedAvg, HARP.")

## 16 — Evaluation metrics

To score the rankings we use five standard information-retrieval metrics, each capturing a different aspect of ranking quality.

### 16.1 — NDCG@k (Normalized Discounted Cumulative Gain)

$$\text{DCG@k} \;=\; \sum_{i=1}^{k} \frac{g_{\pi(i)}}{\log_2(i + 1)}, \qquad \text{NDCG@k} \;=\; \frac{\text{DCG@k}}{\text{IDCG@k}}.$$

Captures overall ordering quality among the top $k$, with a logarithmic discount so the very top positions matter most. NDCG ranges from 0 (worst) to 1 (perfect).

### 16.2 — MRR (Mean Reciprocal Rank)

$$\text{MRR} \;=\; \frac{1}{\text{rank of first relevant agent}}.$$

Sensitive only to **where the first good agent lands**. MRR = 1 if the top-1 is relevant; 0.5 if rank-2 is the first; 0.33 if rank-3, and so on.

### 16.3 — Precision@k

The fraction of the top-$k$ that are relevant. **The most practical metric** for routing: "of the top 3 the system surfaces, how many are actually good?"

### 16.4 — Recall@k

The fraction of *all* relevant agents that appear in the top-$k$. Measures **coverage**.

### 16.5 — Regret

$$\text{Regret} \;=\; \frac{g^\star - g_{\text{chosen}}}{g^\star + \varepsilon}.$$

How much utility we **miss** by picking the top-1 vs the truly best agent. **Lower is better** (the only metric where this is true). Equals 0 when we pick the best, 1 when we pick the worst.

### 16.6 — Ground truth

We need a binary relevance label per (agent, task). We use the simplest principled choice: agents whose mean true skill on the task's required skills is **above the median** are considered "relevant," everyone else not. This makes exactly half the agents relevant on every task, which is the cleanest setup for NDCG/MRR/Precision/Recall to be comparable across tasks.

In [ ]:
def ndcg_at_k(scores: np.ndarray, gains: np.ndarray, k: int = 5) -> float:
    """Normalized Discounted Cumulative Gain at k."""
    order = np.argsort(-scores)[:k]
    dcg = sum(gains[order[i]] / np.log2(i + 2) for i in range(k))
    ideal_order = np.argsort(-gains)[:k]
    idcg = sum(gains[ideal_order[i]] / np.log2(i + 2) for i in range(k))
    return dcg / idcg if idcg > 0 else 0.0


def mean_reciprocal_rank(scores: np.ndarray, gains: np.ndarray) -> float:
    """Reciprocal rank of the first relevant agent in the ranking."""
    for i, ai in enumerate(np.argsort(-scores)):
        if gains[ai] > 0:
            return 1.0 / (i + 1)
    return 0.0


def precision_at_k(scores: np.ndarray, gains: np.ndarray, k: int = 3) -> float:
    """Fraction of the top-k that are relevant."""
    return float(np.mean([gains[ai] > 0 for ai in np.argsort(-scores)[:k]]))


def recall_at_k(scores: np.ndarray, gains: np.ndarray, k: int = 5) -> float:
    """Fraction of all relevant agents that appear in the top-k."""
    total_pos = max(int((gains > 0).sum()), 1)
    return float(np.sum([gains[ai] > 0 for ai in np.argsort(-scores)[:k]])) / total_pos


def regret(scores: np.ndarray, gains: np.ndarray) -> float:
    """Utility missed by picking the top-1 vs the best possible. LOWER is better."""
    chosen = int(np.argmax(scores))
    return float((gains.max() - gains[chosen]) / (gains.max() + 1e-9))


def task_gains(
    task_idx: int,
    true_skill: np.ndarray,
    task_required_skills: List[List[int]],
) -> np.ndarray:
    """
    Binary ground-truth relevance per agent for this task: agents whose
    mean true-skill on the task's required skills is above the median
    are considered 'relevant'.
    """
    skills_i = task_required_skills[task_idx]
    raw = true_skill[:, skills_i].mean(axis=1)
    return (raw > np.median(raw)).astype(float)


print("Evaluation metrics ready: NDCG@5, MRR, P@3, R@5, Regret.")
print(f"Sample ground-truth gains for task {test_idx[0]}: "
      f"{task_gains(test_idx[0], true_skill, task_required_skills)}")

## 17 — The main benchmark

We loop over every **held-out test task**, run each of the six methods, score each ranking with each of the five metrics, and aggregate the means. This is the table you would put in a paper.

You should see HARP at or near the top of every metric **except Regret** (where it should be near the bottom — recall, lower regret is better). If it is not winning, the most common culprits are (a) too few training tasks, so the performance signal is too noisy, or (b) a beta weighting that overcounts a poor-quality signal. In a real deployment you would tune $(\beta_P, \beta_C, \beta_\phi)$ on a held-out slice of the training tasks.

In [ ]:
methods = {
    "Random":      lambda ti: baseline_random(ti),
    "Perf-only":   lambda ti: baseline_perf_only(ti, W_P=W_P),
    "Sim-only":    lambda ti: baseline_sim_only(
                       ti, agent_emb=agent_emb, task_emb=task_emb),
    "VanillaPR":   lambda ti: baseline_vanilla_pagerank(ti, W_C=W_C),
    "WeightedAvg": lambda ti: baseline_weighted_avg(
                       ti, W_P=W_P, W_C=W_C,
                       agent_emb=agent_emb, task_emb=task_emb),
    "HARP":        lambda ti: baseline_harp(ti),
}

metric_fns = {
    "NDCG@5": lambda s, g: ndcg_at_k(s, g, k=5),
    "MRR":    mean_reciprocal_rank,
    "P@3":    lambda s, g: precision_at_k(s, g, k=3),
    "R@5":    lambda s, g: recall_at_k(s, g, k=5),
    "Regret": regret,
}

# Score every method on every test task with every metric.
# Progress bar shows test-task progression; inner loops are too fast to instrument.
results = {m: {k: [] for k in metric_fns} for m in methods}
t0 = time.perf_counter()
for ti in tqdm(test_idx, desc="Benchmarking", disable=not HAVE_TQDM):
    g = task_gains(ti, true_skill, task_required_skills)
    for method_name, method_fn in methods.items():
        scores = method_fn(ti)
        for metric_name, metric_fn in metric_fns.items():
            results[method_name][metric_name].append(metric_fn(scores, g))
elapsed = time.perf_counter() - t0

# Build the summary table.
summary = pd.DataFrame({
    m: {k: float(np.mean(v)) for k, v in d.items()}
    for m, d in results.items()
}).T

# Persist the table so downstream dashboards / CI can compare across runs.
summary_path = CONFIG.artifact_dir / "benchmark_summary.csv"
summary.to_csv(summary_path)
logger.info(
    "Benchmark complete in %.2fs across %d test tasks; written to %s",
    elapsed, len(test_idx), summary_path,
)

print("=" * 70)
print("MAIN BENCHMARK — Mean metric values across test tasks")
print("=" * 70)
print(summary.round(3).to_string())
print()
print("Higher is better for NDCG, MRR, P@3, R@5.")
print("Lower  is better for Regret.")
print(f"\nElapsed: {elapsed:.2f}s for {len(methods)} methods x {len(test_idx)} tasks "
      f"= {len(methods) * len(test_idx)} method-evaluations.")
print(f"Summary written to: {summary_path}")
summary.round(3)

## 18 — Ablation study: which signal carries the load?

A benchmark table tells you *that* HARP wins. An **ablation** tells you **why**. We turn off each signal in turn (set its $\beta$ to zero and redistribute the freed mass to the other two), then re-measure NDCG@5.

We also include a variant that swaps the **semantic teleport** for a uniform teleport — to isolate how much of HARP's win comes from the matrix mix vs how much comes from the teleport personalization.

A healthy ablation should show that **every component matters** — i.e. every `HARP-no-X` variant is meaningfully worse than `HARP-full`. If one component can be removed with no loss, that is informative: you have a redundant signal and could simplify the system.

In [ ]:
def harp_with_uniform_teleport(
    task_idx: int,
    beta: Tuple[float, float, float],
    alpha: float = CONFIG.alpha,
    max_iter: int = CONFIG.max_iter,
    tol: float = CONFIG.tolerance,
) -> np.ndarray:
    """
    HARP variant that uses a UNIFORM teleport instead of the semantic one.

    Used in the ablation to isolate how much of HARP's win comes from the
    convex matrix mix vs the personalized teleport vector.
    """
    M_P, M_C, M_phi, A_idx, *_ = make_transition_matrices(
        task_idx, W_P, W_C, agent_emb, skill_emb, task_emb,
    )
    M = beta[0] * M_P + beta[1] * M_C + beta[2] * M_phi
    V = M.shape[0]
    p = np.ones(V) / V                          # uniform replacement for p_tau
    r = p.copy()
    for _ in range(max_iter):
        r_new = alpha * (M @ r) + (1 - alpha) * p
        if np.linalg.norm(r_new - r, 1) < tol:
            break
        r = r_new
    return r[A_idx]


# Each row: (name, betas, teleport-kind).
# When we zero a beta we redistribute the freed mass so the betas still sum to 1.
ablation_configs: List[Tuple[str, Tuple[float, float, float], str]] = [
    ("HARP-full",              CONFIG.beta,           "softmax"),
    ("HARP-no-Performance",    (0.0, 0.5, 0.5),       "softmax"),
    ("HARP-no-Endorsement",    (0.6, 0.0, 0.4),       "softmax"),
    ("HARP-no-Semantic",       (0.6, 0.4, 0.0),       "softmax"),
    ("HARP-uniform-teleport",  CONFIG.beta,           "uniform"),
]

ablation_rows: List[Dict[str, object]] = []
for name, beta, kind in tqdm(ablation_configs, desc="Ablations", disable=not HAVE_TQDM):
    ndcgs: List[float] = []
    for ti in test_idx:
        g = task_gains(ti, true_skill, task_required_skills)
        if kind == "uniform":
            scores = harp_with_uniform_teleport(ti, beta)
        else:
            scores, _ = harp_rank(
                ti, W_P, W_C, agent_emb, skill_emb, task_emb, beta=beta,
            )
        ndcgs.append(ndcg_at_k(scores, g, k=5))
    ablation_rows.append({
        "Variant": name,
        "beta": str(beta),
        "NDCG@5": float(np.mean(ndcgs)),
    })

ablation_df = pd.DataFrame(ablation_rows)
ablation_path = CONFIG.artifact_dir / "ablation_results.csv"
ablation_df.to_csv(ablation_path, index=False)
logger.info("Ablation results written to %s", ablation_path)

print("=" * 60)
print("ABLATION — Which signal carries the load?")
print("=" * 60)
print(ablation_df.round(3).to_string(index=False))
print()
print("The drop from HARP-full to each ablation tells you")
print("how much that component contributes.")
print(f"Ablation results written to: {ablation_path}")
ablation_df.round(3)

## 19 — Visualizing convergence

Theorem 1 (informal) promised that HARP converges **geometrically** with rate $\alpha$. Let's confirm that empirically by plotting the $\ell_1$ residual at each iteration against the theoretical $\alpha^t$ bound. On a log scale, both curves should be **roughly linear**, and the empirical one should sit at or below the theoretical line.

This plot is the mathematical proof staring at you in graphical form. The dashed line is the **theoretical worst-case** bound; the dots are the **actual behavior**; the gap between them is how much you could in principle tighten the analysis for this specific instance.

In [ ]:
# Run HARP on one task with diagnostics turned on to grab the residual trace.
ti = test_idx[0]
_, n_iters, residuals = harp_rank(
    ti, W_P, W_C, agent_emb, skill_emb, task_emb,
    return_diagnostics=True,
)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(residuals, "o-",
            label=r"HARP residual $\|r_{t+1}-r_t\|_1$",
            markersize=4)
theoretical = [residuals[0] * (CONFIG.alpha ** i) for i in range(len(residuals))]
ax.semilogy(theoretical, "--",
            label=fr"Theory: $\alpha^t$ bound ($\alpha$={CONFIG.alpha})")
ax.axhline(CONFIG.tolerance, color="red", linestyle=":",
           label=f"Tolerance {CONFIG.tolerance}")
ax.set_xlabel("Power iteration step")
ax.set_ylabel("L1 residual (log scale)")
ax.set_title(r"Empirical contraction matches the $\alpha$-rate guarantee")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()

# Persist the figure alongside other artifacts.
convergence_path = CONFIG.artifact_dir / "convergence.png"
fig.savefig(convergence_path, dpi=120, bbox_inches="tight")
logger.info("Convergence plot saved to %s", convergence_path)
plt.show()

print(f"Converged in {n_iters} iterations on task {ti}.")
print(f"Final residual: {residuals[-1]:.2e}  (tolerance was {CONFIG.tolerance})")
print(f"Plot saved to: {convergence_path}")

## 20 — Visualizing the agent-score landscape

A heatmap of HARP scores across **all test tasks and all agents** is the single most informative diagnostic. Each row is a task, each column is an agent, and the cell color is the HARP score.

Things to look for in the resulting plot:

- **Vertical stripes** — some agents always get high scores regardless of the task. Could be a healthy generalist *or* an over-rewarded one. Cross-reference with the agent description and history.
- **Horizontal stripes** — some tasks have a clear single winner. Usually true for L3 tasks with very specific skill requirements.
- **Cold columns** — agents that never win. They are either genuinely weak or specialized for a skill that did not show up in this test slice.

A healthy heatmap has **a mix of both** vertical and horizontal patterns.

In [ ]:
# Build a (n_test_tasks, n_agents) score matrix.
score_matrix = np.zeros((len(test_idx), n_agents))
for row_i, ti in enumerate(tqdm(test_idx, desc="Scoring tasks", disable=not HAVE_TQDM)):
    score_matrix[row_i], _ = harp_rank(
        ti, W_P, W_C, agent_emb, skill_emb, task_emb,
    )

fig, ax = plt.subplots(figsize=(12, 6))
if HAVE_SEABORN:
    sns.heatmap(
        score_matrix,
        xticklabels=AGENT_NAMES,
        yticklabels=[f"t{ti:02d} L{TASKS.iloc[ti].level}" for ti in test_idx],
        cmap="viridis",
        cbar_kws={"label": "HARP score"},
        ax=ax,
    )
else:
    im = ax.imshow(score_matrix, aspect="auto", cmap="viridis")
    fig.colorbar(im, ax=ax, label="HARP score")
    ax.set_xticks(range(n_agents))
    ax.set_xticklabels(AGENT_NAMES, rotation=45, ha="right")
    ax.set_yticks(range(len(test_idx)))
    ax.set_yticklabels([f"t{ti:02d} L{TASKS.iloc[ti].level}" for ti in test_idx])

ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_title("HARP agent scores across test tasks")
ax.set_xlabel("Agent")
ax.set_ylabel("Task (id + level)")
fig.tight_layout()

heatmap_path = CONFIG.artifact_dir / "score_heatmap.png"
fig.savefig(heatmap_path, dpi=120, bbox_inches="tight")
logger.info("Score heatmap saved to %s", heatmap_path)
plt.show()
print(f"Heatmap shape: {score_matrix.shape}  (n_test_tasks x n_agents)")
print(f"Plot saved to: {heatmap_path}")

## 21 — Performance microbenchmark

A production router needs to answer **how long does one HARP ranking take?** This decides whether you can call it synchronously inside a user request (target: < 50 ms) or whether you need an async queue.

We measure two things:

1. **Cold path** — building $\mathbf{M}_\tau$ and running power iteration for one *known* task (a task already in `task_emb`). This is the steady-state cost once the system is warm.
2. **End-to-end path** — embedding a brand-new task description, appending it, then running HARP. This is what a real API endpoint experiences on every request.

We report the median and p95 over 30 runs (median resists outliers; p95 is what your SLO will be set against).

In [ ]:
def _percentile(values: List[float], pct: float) -> float:
    """Plain-python percentile helper (no SciPy needed)."""
    if not values:
        raise ValueError("Cannot compute percentile of empty list.")
    return float(np.percentile(np.asarray(values), pct))


def benchmark_harp(
    n_runs: int = 30,
    new_task_text: str = "Summarize this PDF and extract every cited URL.",
) -> pd.DataFrame:
    """
    Microbenchmark the HARP hot path.

    Returns
    -------
    pd.DataFrame
        One row per scenario (cold / end-to-end) with median and p95
        latencies in milliseconds.
    """
    results: List[Dict[str, object]] = []

    # ---- Scenario 1: cold path on a known task ------------------------
    cold_times: List[float] = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _scores, _ = harp_rank(
            test_idx[0], W_P, W_C, agent_emb, skill_emb, task_emb,
        )
        cold_times.append((time.perf_counter() - t0) * 1000.0)
    results.append({
        "scenario": "rank_known_task (no embed)",
        "n_runs": n_runs,
        "median_ms": float(np.median(cold_times)),
        "p95_ms": _percentile(cold_times, 95),
        "min_ms": float(np.min(cold_times)),
        "max_ms": float(np.max(cold_times)),
    })

    # ---- Scenario 2: end-to-end including embedding -------------------
    e2e_times: List[float] = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = rank_agents_for_new_task(
            new_task_text, embedder, W_P, W_C,
            agent_emb, skill_emb, task_emb,
            top_k=5, verbose=False,
        )
        e2e_times.append((time.perf_counter() - t0) * 1000.0)
    results.append({
        "scenario": "rank_new_task (embed + rank)",
        "n_runs": n_runs,
        "median_ms": float(np.median(e2e_times)),
        "p95_ms": _percentile(e2e_times, 95),
        "min_ms": float(np.min(e2e_times)),
        "max_ms": float(np.max(e2e_times)),
    })

    return pd.DataFrame(results)


# Define rank_agents_for_new_task before benchmarking; it is defined just below
# in Section 23 in the original ordering. To keep this microbenchmark
# self-contained, we re-declare it here using the same logic so the cell
# can run independently.
def rank_agents_for_new_task(
    task_description: str,
    embedder: SentenceTransformer,
    W_P: np.ndarray,
    W_C: np.ndarray,
    agent_emb: np.ndarray,
    skill_emb: np.ndarray,
    task_emb_base: np.ndarray,
    top_k: int = 5,
    verbose: bool = True,
) -> List[Tuple[str, float]]:
    """Embed a new task and rank agents (see Section 23 for full docstring)."""
    if not task_description.strip():
        raise ValueError("task_description must be non-empty.")
    new_emb = embedder.encode(
        [task_description], normalize_embeddings=True, show_progress_bar=False,
    )
    task_emb_extended = np.vstack([task_emb_base, new_emb])
    new_idx = len(task_emb_extended) - 1
    scores, n_iters = harp_rank(
        new_idx, W_P, W_C, agent_emb, skill_emb, task_emb_extended,
    )
    order = np.argsort(-scores)
    if verbose:
        print(f"\nTask: \"{task_description}\"")
        print(f"Converged in {n_iters} iterations. Top {top_k} agents:")
        print("-" * 78)
        for rank, ai in enumerate(order[:top_k], 1):
            print(f"  #{rank}: {AGENT_NAMES[ai]:35s}  score={scores[ai]:.4f}")
    return [(AGENT_NAMES[ai], float(scores[ai])) for ai in order[:top_k]]


perf_df = benchmark_harp(n_runs=30)
perf_path = CONFIG.artifact_dir / "performance.csv"
perf_df.to_csv(perf_path, index=False)
logger.info("Performance benchmark written to %s", perf_path)

print("=" * 70)
print("PERFORMANCE MICROBENCHMARK")
print("=" * 70)
print(perf_df.round(2).to_string(index=False))
print()
print("Use p95_ms to set your SLO.  cold path is what a warm API endpoint sees;")
print("end-to-end includes embedding the new task text.")
perf_df.round(2)

## 22 — Persisting artifacts (snapshot the model state)

In a production deployment HARP runs as a long-lived service. You **do not** want to rebuild $W^P$, $W^C$, and the three embedding matrices from scratch every restart — that means re-running the history simulator and re-embedding everything, which costs minutes and is non-deterministic if your history source has changed.

The right pattern is:

1. On a cron job (say nightly), refresh $W^P$ and $W^C$ from the latest invocation logs.
2. **Snapshot the new state to disk** — embeddings, weight matrices, and a manifest that records which CONFIG / data version produced them.
3. The serving process loads the snapshot at startup; ranking requests then run with sub-millisecond setup cost.

This cell demonstrates the snapshot side. The `load_snapshot()` helper below shows the inverse — useful for warm-restart testing or for shipping a "frozen" model with your service.

In [ ]:
import dataclasses
from datetime import datetime, timezone


def save_snapshot(
    snapshot_dir: Path,
    *,
    config: HARPConfig,
    W_P: np.ndarray,
    W_C: np.ndarray,
    agent_emb: np.ndarray,
    skill_emb: np.ndarray,
    task_emb: np.ndarray,
    agent_names: List[str],
    skill_names: List[str],
    extra_meta: Optional[Dict[str, str]] = None,
) -> Path:
    """
    Serialize a complete HARP state snapshot to disk.

    Layout under ``snapshot_dir``:
        manifest.json          - HARPConfig, schema_version, timestamp, sizes
        W_P.npy, W_C.npy       - weight matrices
        agent_emb.npy          - (n_agents, d) embedding matrix
        skill_emb.npy          - (n_skills, d) embedding matrix
        task_emb.npy           - (n_tasks, d)  embedding matrix
        agents.txt, skills.txt - identifiers (one per line, indexed)

    Returns
    -------
    Path
        The snapshot directory (created if it did not already exist).
    """
    snapshot_dir.mkdir(parents=True, exist_ok=True)

    manifest: Dict[str, object] = {
        "schema_version": 1,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "config": {k: (str(v) if isinstance(v, Path) else v)
                   for k, v in dataclasses.asdict(config).items()},
        "sizes": {
            "n_agents": int(agent_emb.shape[0]),
            "n_skills": int(skill_emb.shape[0]),
            "n_tasks": int(task_emb.shape[0]),
            "embedding_dim": int(agent_emb.shape[1]),
            "W_P_shape": list(W_P.shape),
            "W_C_shape": list(W_C.shape),
        },
        "extra": extra_meta or {},
    }
    (snapshot_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))

    np.save(snapshot_dir / "W_P.npy", W_P)
    np.save(snapshot_dir / "W_C.npy", W_C)
    np.save(snapshot_dir / "agent_emb.npy", agent_emb)
    np.save(snapshot_dir / "skill_emb.npy", skill_emb)
    np.save(snapshot_dir / "task_emb.npy", task_emb)
    (snapshot_dir / "agents.txt").write_text("\n".join(agent_names))
    (snapshot_dir / "skills.txt").write_text("\n".join(skill_names))

    return snapshot_dir


def load_snapshot(snapshot_dir: Path) -> Dict[str, object]:
    """
    Load a HARP snapshot back from disk.

    Returns
    -------
    Dict[str, object]
        Keys: ``manifest``, ``W_P``, ``W_C``, ``agent_emb``, ``skill_emb``,
        ``task_emb``, ``agent_names``, ``skill_names``.

    Raises
    ------
    FileNotFoundError
        If ``snapshot_dir / manifest.json`` is missing.
    """
    manifest_path = snapshot_dir / "manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"No HARP snapshot found at {snapshot_dir}")

    manifest = json.loads(manifest_path.read_text())
    return {
        "manifest": manifest,
        "W_P": np.load(snapshot_dir / "W_P.npy"),
        "W_C": np.load(snapshot_dir / "W_C.npy"),
        "agent_emb": np.load(snapshot_dir / "agent_emb.npy"),
        "skill_emb": np.load(snapshot_dir / "skill_emb.npy"),
        "task_emb": np.load(snapshot_dir / "task_emb.npy"),
        "agent_names": (snapshot_dir / "agents.txt").read_text().splitlines(),
        "skill_names": (snapshot_dir / "skills.txt").read_text().splitlines(),
    }


# Persist the current state under a timestamped folder so multiple runs
# do not clobber each other in CI.
_snapshot_ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
_snapshot_dir = CONFIG.artifact_dir / f"snapshot_{_snapshot_ts}"

save_snapshot(
    _snapshot_dir,
    config=CONFIG,
    W_P=W_P, W_C=W_C,
    agent_emb=agent_emb, skill_emb=skill_emb, task_emb=task_emb,
    agent_names=AGENT_NAMES, skill_names=SKILLS,
    extra_meta={"history_rows": str(len(H)),
                "train_tasks": str(len(train_idx)),
                "test_tasks": str(len(test_idx))},
)
logger.info("Snapshot written to %s", _snapshot_dir)

# Round-trip sanity check: reload and assert numerical equality.
_loaded = load_snapshot(_snapshot_dir)
assert np.array_equal(_loaded["W_P"], W_P), "W_P round-trip failed"
assert np.array_equal(_loaded["W_C"], W_C), "W_C round-trip failed"
assert _loaded["agent_names"] == AGENT_NAMES, "agent_names round-trip failed"
logger.info("Snapshot round-trip verified.")

print("=" * 70)
print("HARP STATE SNAPSHOT")
print("=" * 70)
print(f"Snapshot dir: {_snapshot_dir}")
for item in sorted(_snapshot_dir.iterdir()):
    size_kb = item.stat().st_size / 1024
    print(f"  {item.name:24s}  {size_kb:8.1f} KB")
print("\nLoad later with:")
print("    from pathlib import Path")
print(f"    state = load_snapshot(Path('{_snapshot_dir}'))")
print("    rank_agents_for_new_task(task, embedder, state['W_P'], state['W_C'],")
print("        state['agent_emb'], state['skill_emb'], state['task_emb'])")

## 23 — Interactive demo: rank agents for *your* task

This is the part that makes the notebook an **application** rather than an experiment. We expose a single function — `rank_agents_for_new_task(...)` — that takes a free-text task description and returns the top-$k$ agents per HARP. **This is exactly the API endpoint you would wire into a production router.**

Under the hood it:

1. Embeds the new task with the same `all-MiniLM-L6-v2` model.
2. Appends the embedding to the existing `task_emb` matrix (temporarily) so the rest of the pipeline can index it.
3. Runs `harp_rank` on the new task index.
4. Returns the ranked agent names with their scores.

The function takes constant time per call regardless of how many tasks have already been processed — embeddings are cached and the matrices are precomputed.

You should see, qualitatively:

- A **python-scraping task** ranks `AutoGen-Coder` or `SWE-Agent-Claude` near the top.
- A **PDF-summarization task** ranks the deep-research agents highly.
- An **image-receipt task** favors agents whose description mentions vision.
- An **audio-transcription task** ranks specialist transcribers (if any) ahead of pure coders.

If you see weird mismatches, that is **informative** — it usually means either the agent description is too vague or the history is too sparse for that skill category.

In [ ]:
def rank_agents_for_new_task(
    task_description: str,
    embedder: SentenceTransformer,
    W_P: np.ndarray,
    W_C: np.ndarray,
    agent_emb: np.ndarray,
    skill_emb: np.ndarray,
    task_emb_base: np.ndarray,
    top_k: int = 5,
    verbose: bool = True,
) -> List[Tuple[str, float]]:
    """
    Production-shaped HARP entry point: rank agents for a fresh task description.

    This is the function you would expose as your routing API endpoint. It is
    pure (no global mutation), input-validated, and emits structured logs at
    DEBUG level so you can trace per-request latency from the log stream alone.

    Parameters
    ----------
    task_description : str
        Free-text task description. Must be non-empty after stripping.
    embedder : SentenceTransformer
        Embedder used to produce ``agent_emb`` / ``skill_emb`` / ``task_emb_base``.
    W_P : (n_agents, n_skills) ndarray
    W_C : (n_agents, n_agents) ndarray
    agent_emb, skill_emb, task_emb_base : ndarray
        Precomputed embedding matrices. ``task_emb_base`` does NOT need to
        contain the new task; we append it in a temporary, request-local copy.
    top_k : int
        How many agents to return (must be >= 1 and <= n_agents).
    verbose : bool
        If True, also pretty-print the ranking to stdout.

    Returns
    -------
    List[Tuple[str, float]]
        Top-k (agent_name, score) pairs ordered by descending score.

    Raises
    ------
    ValueError
        On empty task_description or out-of-range top_k.
    """
    if not task_description or not task_description.strip():
        raise ValueError("task_description must be non-empty.")
    n_a = len(AGENT_NAMES)
    if not (1 <= top_k <= n_a):
        raise ValueError(f"top_k must be in [1, {n_a}]; got {top_k}")

    t_start = time.perf_counter()

    # 1. Embed the new task with the same model used for the corpus.
    new_emb = embedder.encode(
        [task_description], normalize_embeddings=True, show_progress_bar=False,
    )
    t_embed = time.perf_counter()

    # 2. Append (request-local copy; do NOT mutate the shared task_emb_base).
    task_emb_extended = np.vstack([task_emb_base, new_emb])
    new_idx = len(task_emb_extended) - 1

    # 3. Score with HARP.
    scores, n_iters = harp_rank(
        new_idx, W_P, W_C, agent_emb, skill_emb, task_emb_extended,
    )
    t_rank = time.perf_counter()

    # 4. Format output.
    order = np.argsort(-scores)
    result = [(AGENT_NAMES[ai], float(scores[ai])) for ai in order[:top_k]]

    logger.debug(
        "rank_agents_for_new_task: embed=%.1fms rank=%.1fms total=%.1fms iters=%d top1=%s",
        (t_embed - t_start) * 1000,
        (t_rank - t_embed) * 1000,
        (t_rank - t_start) * 1000,
        n_iters,
        result[0][0],
    )

    if verbose:
        print(f"\nTask: \"{task_description}\"")
        print(f"Converged in {n_iters} iterations. Top {top_k} agents:")
        print("-" * 78)
        for rank, ai in enumerate(order[:top_k], 1):
            print(f"  #{rank}: {AGENT_NAMES[ai]:35s}  score={scores[ai]:.4f}")
    return result


# Demonstrate on four diverse tasks that exercise different skill categories.
example_tasks = [
    "Write a python script that scrapes the top stories from Hacker News and saves them to a CSV.",
    "Read this academic PDF and summarize the methodology section in three bullet points.",
    "Open the attached image of a receipt and extract the total amount and merchant name.",
    "Listen to this podcast clip and transcribe the first two minutes.",
]

print("=" * 78)
print("INTERACTIVE DEMO — Ranking agents for arbitrary new task descriptions")
print("=" * 78)
for t in example_tasks:
    rank_agents_for_new_task(
        t, embedder, W_P, W_C, agent_emb, skill_emb, task_emb, top_k=3,
    )

## 24 — Try it yourself

Edit the `MY_TASK` string in the cell below to anything you like, then re-run. The full HARP ranker will execute on your task description. No re-training required, no GPU needed — under a second per call.

In [ ]:
MY_TASK = (
    "Find the latest annual revenue of OpenAI from a recent news article, "
    "open the linked PDF report, and produce a 3-sentence summary."
)

ranked = rank_agents_for_new_task(
    MY_TASK, embedder, W_P, W_C, agent_emb, skill_emb, task_emb, top_k=5,
)

## 25 — What you have built, and what to do next

You now have a **production-grade agent ranker** that fuses three signals — performance history, endorsement structure, and semantic similarity — into a single principled score, with provable convergence and a clean cold-start strategy. End-to-end it runs in well under a minute on Colab's free tier, and on a warm process each request takes single-digit milliseconds.

### The whole pipeline in one diagram

```
agents  ──┐                                                    ┌─→  W^P  ─→  M^P  ─┐
skills  ──┤  (sentence embeddings)  ─→  (simulate history)  ─→─┤                   │
tasks   ──┘                                                    └─→  W^C  ─→  M^C  ─┤
                                                                                   ├─→  M_τ  ──→  power iteration  ──→  agent ranking
                  (cosine-sim against task)  ───────────→  M^φ  ───────────────────┘                  │
                                                                                                      │
                  softmax(κ · cos)  ───────────────→  p_τ  (teleport)  ───────────────────────────────┘
```

### Production hardening checklist (what we did in this notebook)

| Concern | How it is handled here |
|---|---|
| **Pinned dependencies** | Major-version pins in the install cell (Section 2). |
| **Structured logging** | `logging` module with `HARP_LOG_LEVEL` env var (Section 3). |
| **Validated configuration** | Immutable `HARPConfig` dataclass with `__post_init__` invariants. |
| **Reproducibility** | Single `CONFIG.seed`, environment manifest written to disk (3a). |
| **Optional-dep guards** | Graceful fallback when `seaborn` / `tqdm` are missing. |
| **Disk caching** | Content-hashed `embed_with_cache` (Section 7) — instant rerun. |
| **Vectorized hot loops** | `groupby` for $W^P$, broadcast outer-products for $W^C$, `np.ix_` block fills for $M^P$/$M^C$ (Sections 9, 10, 12). |
| **Numerical stability** | Subtract-max softmax in teleport (Section 13). |
| **Input validation** | `ValueError`-raising guards on every public function. |
| **Runtime invariants** | `assert`s for column-stochasticity, non-negativity, in-range. |
| **Observability** | `tqdm` progress bars on every long loop, per-request latency logs. |
| **Persistence** | `save_snapshot` / `load_snapshot` for warm-restart (Section 22). |
| **SLO measurement** | `benchmark_harp` with median + p95 latency (Section 21). |

### Natural extensions, roughly in order of effort

| Extension | Effort | Payoff |
|---|---|---|
| Replace synthetic tasks with real **GAIA** via `datasets.load_dataset` | small | real-world numbers instead of synthetic |
| Add a **Thompson-sampling** exploration layer on top of HARP scores | small | online deployment with controlled exploration |
| Swap the sentence transformer for **BGE-M3** or a stronger reranker | small | sharper semantic signal, higher precision |
| **Time-decay** the history so recent outcomes dominate stale ones | medium | adapts as your agent pool evolves |
| Replace co-success $W^C$ with **explicit invocation logs** from your router | medium | sharper endorsement signal |
| Add a **per-skill latency / cost / risk** penalty layer | medium | production-grade SLO compliance |
| Wrap the ranker as a **FastAPI** service with auth / rate-limiting | medium | drop-in HTTP endpoint |
| **Federated** trust aggregation across multiple tenants | large | privacy-preserving multi-tenant deployments |
| Tensor extension giving up uniqueness for **higher-order Markov structure** | large | true paper waiting to be written |

### Operating the notebook as a service

The `save_snapshot` / `load_snapshot` pair in Section 22 is the contract between **batch** and **serving** processes:

- **Batch (nightly cron)**: re-run Sections 6 → 22 on the latest invocation logs, then `save_snapshot(prod_dir, ...)`.
- **Serving (long-running process)**: at startup, `state = load_snapshot(prod_dir)`. For every request, call `rank_agents_for_new_task(...)` with `state[...]` arrays. No state mutation, no global state, safe to scale horizontally.

### Files in this codebase

The reference implementations in `agent_ranking_colab_codebases/`:

- `harp.py` — the script form of this notebook with a CLI (`python harp.py --task "..."`).
- `research_agent_skill_rank.py` — a sibling research codebase (AURA-Rank) that fuses HARP-like graph authority with permission/auth/cost/latency/risk features for a real MCP gateway.
- `real_usecase_mcp_router.py` — the SaaS backend prototype built on the same primitives.

Artifacts written by this notebook (under `CONFIG.artifact_dir`, default `.harp_cache/`):

- `environment.json` — runtime metadata.
- `emb_*.npy` — content-hashed embedding caches.
- `benchmark_summary.csv` — main benchmark table.
- `ablation_results.csv` — ablation table.
- `performance.csv` — latency microbenchmark.
- `convergence.png`, `score_heatmap.png` — diagnostic plots.
- `snapshot_<timestamp>/` — complete state snapshot.

### Citation

If you use HARP in your work, please cite the original writeup in `documentation/AgentRank and SkillRank for a Universal MCP Gateway.docx` and the convergence proofs in `documentation/HARP alogrithm.pdf`.

### License

This implementation is released as a research prototype for the AgentHub project. See the repository LICENSE file for terms.

---

**Run order recap**: Sections 2 → 3 → 3a → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13 → 14 → 15 → 16 → 17 → 18 → 19 → 20 → 21 → 22 → 23 → 24. Each section's code cell depends on every prior section having been run.